# Projeto — Que som urbano é este? Classificação de áudio com CNN (UrbanSound8K)

**Disciplina 3 · Deep Learning · IASEG · Prof. Rafael Stutz** · Grupo: Lucas Hot, Marlon, Jonathan e Tiago

**Pergunta:** que som urbano está presente neste trecho de áudio? Com atenção especial a **disparos de arma de fogo** (`gun_shot`).

| | |
|---|---|
| **Entrada** | um trecho de áudio de até 4 s, transformado em espectrograma de mel (uma "imagem" de 64 × 173) |
| **Saída** | 10 probabilidades, uma por classe (somam 1) |
| **Tarefa** | classificação **multiclasse**: cada trecho tem uma classe só |
| **Métrica principal** | acurácia na validação **sem atalho**, com recall e precisão do tiro ao lado |

| Seção do relatório | Etapas do notebook |
|---|---|
| 1 · Objetivo da modelagem | 1 |
| 2 · Dados utilizados | 2, 3, 4, 5 |
| 3 · Arquitetura e desempenho | 6 a 12 |
| 4 · Análise do treinamento | 7 a 11 |
| 5 · Limitações | 5, 13 (validação cruzada em 13.6; mudança de distribuição em 13.7; Grad-CAM e intervenção em 13.8 e 13.9) |
| 6 · Uso pretendido e restrições | 11.5, 11.6, 11.9, 11.12, 13.7, 14, 15 |

**O caminho dos experimentos** (tudo escolhido na validação, fold 9; o teste, fold 10, só na Etapa 12):

| Rodada | O que se testa | Como se compara |
|---|---|---|
| CNN 1 a 4 (Etapas 7 a 10) | primeira rede, dropout, entrada sem atalho, canal de delta | uma semente |
| Rodada 1 → **CNN 5** (11.1, 11.2) | cinco mudanças, uma de cada vez, em cima da CNN 4 | uma semente; a vencedora é repetida com 3 sementes |
| Rodada 2 → **CNN 6** (11.3) | três mudanças em cima da CNN 5 | **3 sementes cada**, pela média |
| Rodada 3 → **CNN 7** (11.4) | três mudanças em cima da CNN 6 | 3 sementes cada, pela média |
| Rodada 4 → **CNN 8** (11.5) | treinar também com tiros somados a outros sons | 3 sementes; critério do tiro com outro som, declarado antes do teste |
| Rodada 5 → **CNN 9, a final** (11.6) | três aumentos de dados (misturas sem tiro, bordas suavizadas, microfones simulados) e a combinação das que passaram | 3 sementes; regra declarada antes de treinar |
| Rodada 6 (11.7) | transferência de aprendizado com a YAMNet, rede pré-treinada em áudio | 3 sementes; não substituiu a CNN 9 |
| Rodada 7 (11.8) | RawBoost no lugar dos microfones simulados | 3 sementes; não substituiu |
| Rodada 8 (11.9) | RawBoost junto com os microfones simulados | 3 sementes; critério de não piorar o que estava validado |
| Validação cruzada (13.6) | a receita final, com cada um dos 10 folds como teste | média e desvio nos 10 folds |
| Mudança de distribuição (13.7) | outros microfones | só medida, nada é escolhido |
| O que a rede olhou (13.8 e 13.9) | Grad-CAM e um teste de intervenção (Aula 5) | só análise, nada é escolhido |

**Como rodar no Google Colab**
1. *Ambiente de execução → Alterar o tipo de ambiente de execução → **GPU (T4)***.
2. Na **primeira vez**, o notebook baixa a base (≈ 6 GB) e calcula os espectrogramas (≈ 10 a 15 min). O resultado fica salvo no **Google Drive**, junto com as redes treinadas.
3. Nas **vezes seguintes**, ele carrega tudo do Drive. Para treinar as redes de novo, mude `RETREINAR = True` na Etapa 7.
   - Treinar tudo do zero leva cerca de 2 h com GPU (são cerca de 57 redes, contando as sementes), mais uns 70 min para preparar os dados das rodadas 5 a 8 (Etapas 11.6 a 11.9). A validação cruzada (Etapa 13.6) fica desligada por padrão: mais 20 a 30 min.
4. Rodem as células **na ordem**, de cima para baixo.

## Etapa 0 — Configuração

Bibliotecas das aulas: **numpy** e **pandas** (tabelas), **matplotlib** e **seaborn** (gráficos), **scikit-learn** (baseline e métricas, como na disciplina de ML) e **keras** (a rede).

A novidade é a **librosa**, uma biblioteca para ler áudio e calcular espectrogramas. Usamos poucas funções dela, explicadas na Etapa 4.

In [ ]:
import os, json
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"    # esconde mensagens técnicas do TensorFlow
import numpy as np                      # contas com tabelas de números
import pandas as pd                     # a tabela de metadados
import matplotlib.pyplot as plt         # gráficos
import seaborn as sns                   # gráficos estatísticos
import librosa                          # ler áudio e calcular espectrogramas
import keras                            # a rede neural
from keras import layers                # Conv2D, MaxPooling2D, Dense, Dropout...
from IPython.display import Audio, display   # ouvir os áudios dentro do notebook
from tqdm.auto import tqdm              # barra de progresso nos laços demorados

# Os pesos da rede começam sorteados. Fixar a semente faz os sorteios se repetirem (Aula 4).
keras.utils.set_random_seed(42)

import tensorflow as tf, warnings
tf.get_logger().setLevel("ERROR")
warnings.filterwarnings("ignore", message="Skipping variable loading")   # aviso inofensivo ao carregar pesos salvos
print("Keras:", keras.__version__, "| librosa:", librosa.__version__)
print("GPU disponível:", tf.config.list_physical_devices("GPU") or "não (roda na CPU; no Colab, ative a GPU no menu)")

# Toda figura mostrada também é salva em ./resultados, para o relatório (a Etapa 15 junta tudo num .zip)
PASTA_RESULTADOS = "resultados"
os.makedirs(PASTA_RESULTADOS, exist_ok=True)
_mostrar = plt.show
_numero_figura = [0]
def mostrar_e_salvar(*args, **kwargs):
    _numero_figura[0] += 1
    eixos = plt.gcf().axes
    titulo = (plt.gcf()._suptitle.get_text() if plt.gcf()._suptitle else (eixos[0].get_title() if eixos else ""))
    nome = "".join(c if c.isascii() and c.isalnum() else "_" for c in titulo.lower())[:40].strip("_")
    plt.savefig(os.path.join(PASTA_RESULTADOS, f"fig{_numero_figura[0]:02d}_{nome}.png"), dpi=110, bbox_inches="tight")
    _mostrar(*args, **kwargs)
plt.show = mostrar_e_salvar

## Etapa 1 — Definição do problema · Seção 1

**O problema, em uma frase:** identificar, em um trecho curto de áudio gravado na cidade, qual de 10 sons urbanos está presente, com foco em reconhecer **disparos de arma de fogo**.

**Por que classificação multiclasse?** A resposta é uma classe de uma lista fixa (10 sons), e cada trecho da base tem **uma classe só**. Por isso a saída é `Dense(10, activation="softmax")`: 10 probabilidades que somam 1, e a classe prevista é a de maior probabilidade (Aula 3). A perda é `"sparse_categorical_crossentropy"`, porque os rótulos são inteiros de 0 a 9.

**Por que deep learning?** Áudio é um dado **não estruturado** (Aula 1): a gravação é uma sequência de milhares de números (amostras de som), e nenhum deles, sozinho, tem significado. Ninguém consegue escrever à mão as regras que separam um tiro de um latido. Uma rede convolucional aprende essas representações a partir dos exemplos, olhando o espectrograma como uma imagem.

**Aplicação:** apoio a centrais de monitoramento, priorizando trechos de áudio que parecem disparos para **conferência humana** (Etapa 15). Não é para acionar uma viatura sozinha.

**O que o modelo não faz:** reconhecer vários sons **ao mesmo tempo** no mesmo trecho. Isso seria multirrótulo (uma sigmoide por classe, Aula 3), e a base não tem esse tipo de rótulo. Sons diferentes **em momentos diferentes** de uma gravação longa, ele reconhece analisando a gravação em janelas (Etapa 14.2).

## Etapa 2 — Os dados · Seção 2

**UrbanSound8K** (Salamon, Jacoby e Bello, 2014): 8.732 trechos de até 4 s, cortados de gravações do site Freesound, em 10 classes. Link: https://urbansounddataset.weebly.com/urbansound8k.html. Download público no Zenodo.

### 2.1 Onde guardar os arquivos

O áudio é grande (≈ 6 GB) e calcular os espectrogramas demora. Por isso:
- os **áudios** ficam no disco temporário do Colab (`/content`), que some quando a sessão termina;
- os **espectrogramas prontos**, a tabela de metadados e as **redes treinadas** ficam no **Google Drive**.

O notebook continua reprodutível: quem rodar do zero baixa a base do link público. O Drive é só um atalho de tempo.

In [ ]:
USAR_DRIVE = True   # mude para False se não quiser usar o Google Drive (aí tudo é recalculado a cada sessão)

try:
    from google.colab import drive          # só existe dentro do Colab
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

if NO_COLAB and USAR_DRIVE:
    drive.mount("/content/drive")           # o Colab pede autorização na primeira vez
    PASTA_CACHE = "/content/drive/MyDrive/projeto_us8k"
else:
    PASTA_CACHE = "./cache_us8k"

PASTA_AUDIO = "/content" if NO_COLAB else "."
BASE = os.path.join(PASTA_AUDIO, "UrbanSound8K")              # onde os áudios ficam depois de extraídos
ARQ_CACHE = os.path.join(PASTA_CACHE, "us8k_mel64_v2.npz")    # espectrogramas prontos
ARQ_META = os.path.join(PASTA_CACHE, "UrbanSound8K.csv")      # cópia da tabela de metadados
ARQ_MISTURAS = os.path.join(PASTA_CACHE, "us8k_misturas_tiro_v2.npz")   # tiros somados a outros sons (Etapa 4.2)
PASTA_MODELOS = os.path.join(PASTA_CACHE, "modelos")          # redes treinadas
os.makedirs(PASTA_MODELOS, exist_ok=True)

print("Cache (espectrogramas):", ARQ_CACHE, "| já existe?", os.path.exists(ARQ_CACHE))
print("Áudios:", BASE, "| já existe?", os.path.exists(BASE))

### 2.2 Baixar a base (só na primeira vez)

Se os espectrogramas já estiverem no Drive, esta célula não baixa nada.

In [ ]:
URL = "https://zenodo.org/records/1203745/files/UrbanSound8K.tar.gz?download=1"
ARQ_TAR = os.path.join(PASTA_AUDIO, "UrbanSound8K.tar.gz")

if os.path.exists(ARQ_CACHE) and os.path.exists(ARQ_META) and os.path.exists(ARQ_MISTURAS):
    print("Espectrogramas já calculados: não é preciso baixar os áudios.")
    print("(As células que tocam o áudio original serão puladas.)")
elif os.path.exists(BASE):
    print("Áudios já extraídos em", BASE)
else:
    print("Baixando a base (≈ 6 GB)...")
    import urllib.request, tarfile      # Python puro: funciona no Colab e no Windows
    urllib.request.urlretrieve(URL, ARQ_TAR)
    print("Extraindo...")
    with tarfile.open(ARQ_TAR) as t:
        t.extractall(PASTA_AUDIO)
    os.remove(ARQ_TAR)
    print("Pronto:", os.listdir(BASE))

### 2.3 A tabela de metadados

Cada linha é um trecho de áudio. As colunas:

| Coluna | O que é | Entra na rede? |
|---|---|---|
| `slice_file_name` | nome do arquivo `.wav` | não: só para achar o arquivo |
| `fsID` | a **gravação original** no Freesound de onde o trecho foi cortado | não: é um identificador |
| `start`, `end` | início e fim do trecho na gravação original (em segundos) | não |
| `salience` | 1 = som em primeiro plano; 2 = som ao fundo | não: é uma anotação humana |
| `fold` | o grupo oficial (1 a 10) em que o trecho foi colocado | não: serve para dividir |
| `classID`, `class` | a classe: número de 0 a 9 e nome | é a **resposta** |

**A rede não recebe nenhuma dessas colunas:** a entrada é só o áudio. Um microfone na rua só entrega o som; usar qualquer coluna da tabela seria **vazamento** (Aula 2). Criamos uma coluna a mais, `duracao = end − start`, só para explorar os dados.

In [ ]:
ARQ_META_ORIGINAL = os.path.join(BASE, "metadata", "UrbanSound8K.csv")
if os.path.exists(ARQ_META_ORIGINAL):
    meta = pd.read_csv(ARQ_META_ORIGINAL)
    meta.to_csv(ARQ_META, index=False)          # guarda uma cópia no Drive
else:
    meta = pd.read_csv(ARQ_META)                # usa a cópia guardada

meta["duracao"] = meta["end"] - meta["start"]

# Os nomes das 10 classes, na ordem do classID (0 a 9)
CLASSES = meta.drop_duplicates("classID").sort_values("classID")["class"].tolist()
TIRO = CLASSES.index("gun_shot")                # o classID do tiro (6)

# Nomes em português, para gráficos e tabelas (na ordem do classID)
NOMES_PT = {"air_conditioner": "ar-condicionado", "car_horn": "buzina", "children_playing": "crianças brincando",
            "dog_bark": "latido", "drilling": "furadeira", "engine_idling": "motor em marcha lenta",
            "gun_shot": "disparo de arma", "jackhammer": "britadeira", "siren": "sirene", "street_music": "música de rua"}
ROTULOS = [NOMES_PT[c] for c in CLASSES]

print("Trechos:", len(meta), "| colunas:", meta.shape[1])
print("Valores faltando:", meta.isna().sum().sum())
print("Classes:", dict(enumerate(CLASSES)))
meta.head()

### 2.4 Quantos trechos de cada classe?

In [ ]:
contagem = meta["class"].value_counts().sort_values()
cores = ["#C62828" if c == "gun_shot" else "#90A4AE" for c in contagem.index]

plt.figure(figsize=(8, 4))
plt.barh([NOMES_PT[c] for c in contagem.index], contagem.values, color=cores)
for i, v in enumerate(contagem.values):
    plt.text(v + 10, i, str(v), va="center", fontsize=9)
plt.xlabel("trechos de áudio")
plt.title("Trechos por classe (em vermelho, o tiro)")
plt.show()

print(f"Tiro: {contagem['gun_shot']} trechos = {contagem['gun_shot'] / len(meta):.1%} da base")

**Leitura:** sete classes têm 1.000 trechos; **o tiro é a menor classe** (≈ 4%). A acurácia geral pode ficar alta mesmo errando muitos tiros, porque eles pesam pouco no total. Por isso, ao lado da acurácia, acompanhamos o **recall do tiro** (dos tiros reais, quantos a rede achou) e a **precisão do tiro** (dos que a rede chamou de tiro, quantos eram), como no caso do trote da Aula 3.

### 2.5 Duração, saliência e gravações de origem

- **Os trechos têm a mesma duração?** A rede precisa de entradas do mesmo tamanho.
- **O som está em primeiro plano ou ao fundo?** Sons ao fundo devem ser mais difíceis.
- **Quantas gravações diferentes há por classe?** Muitos trechos da mesma gravação compartilham o mesmo microfone e o mesmo ruído de fundo.

In [ ]:
por_classe = meta.groupby("class").agg(
    trechos=("class", "size"),
    duracao_media_s=("duracao", "mean"),
    pct_menos_de_4s=("duracao", lambda d: 100 * (d < 3.99).mean()),
    pct_ao_fundo=("salience", lambda s: 100 * (s == 2).mean()),
    gravacoes=("fsID", "nunique"),
    max_trechos_mesma_gravacao=("fsID", lambda f: f.value_counts().max()),
).sort_values("duracao_media_s").round(1)
display(por_classe)

ordem = por_classe.index.tolist()
plt.figure(figsize=(9, 4))
sns.boxplot(data=meta, y="class", x="duracao", order=ordem, color="#90A4AE")
plt.xlabel("duração do trecho (s)"); plt.ylabel("")
plt.title("Duração dos trechos por classe")
plt.show()

**Leitura:**
- **O tiro é curto:** em média 1,6 s, e quase todos têm menos de 4 s. Ar-condicionado, música de rua e motor têm quase sempre 4 s. Isso vai virar um problema (Etapa 5).
- **Saliência:** a maioria dos tiros está em primeiro plano (sons limpos e altos). Sirene e buzina aparecem muito ao fundo.
- **Gravações de origem:** algumas classes têm poucas gravações diferentes (veja a coluna `gravacoes`: britadeira e ar-condicionado têm poucas dezenas; latido, centenas); uma única gravação pode render dezenas de trechos. Classes com poucas gravações tendem a ser as mais instáveis de um fold para outro (Etapa 13.6). Se a mesma gravação estivesse no treino e no teste, a rede poderia reconhecer **a gravação**, e não o som. É por isso que usamos os folds oficiais (Etapa 3).

## Etapa 3 — Divisão em treino, validação e teste · Seção 2

A base já vem dividida em **10 folds oficiais**, montados de modo que os trechos de uma mesma gravação fiquem juntos. A descrição do projeto exige respeitar essa divisão.

| Conjunto | Folds | Para quê |
|---|---|---|
| **Treino** | 1 a 8 | ajustar os pesos |
| **Validação** | 9 | comparar configurações e decidir quando parar (`EarlyStopping`) |
| **Teste** | 10 | avaliar o modelo escolhido, **uma vez**, no fim |

**Por que não sortear?** Se misturássemos os trechos e sorteássemos 80/10/10, trechos da mesma gravação cairiam no treino e no teste. A rede reconheceria a gravação (o microfone, o ruído de fundo) e o resultado sairia otimista: é **vazamento** (Aula 2). Notebooks públicos que sorteiam chegam a ~90% de acurácia; com os folds, CNNs simples costumam ficar bem abaixo disso.

**Por que não `validation_split`?** O `validation_split` do Keras separa os últimos 20% do treino, sem olhar os folds. Usamos `validation_data` com o fold 9.

**E se o fold 10 for mais fácil ou mais difícil que os outros?** Os autores da base recomendam a validação cruzada pelos 10 folds. Ela é cara para comparar muitas redes, então a usamos só no fim, para medir a receita escolhida (Etapa 13.6).

In [ ]:
treino = (meta["fold"] <= 8).values
valid = (meta["fold"] == 9).values
teste = (meta["fold"] == 10).values

y_treino = meta.loc[treino, "classID"].values
y_valid = meta.loc[valid, "classID"].values
y_teste = meta.loc[teste, "classID"].values

print(f"Treino: {treino.sum()} | Validação: {valid.sum()} | Teste: {teste.sum()}")
print(f"Tiros — treino: {(y_treino == TIRO).sum()} | validação: {(y_valid == TIRO).sum()} | teste: {(y_teste == TIRO).sum()}")

# Conferência do vazamento: alguma gravação original aparece em mais de um conjunto?
g_tr, g_va, g_te = (set(meta.loc[m, "fsID"]) for m in (treino, valid, teste))
print("Gravações em comum — treino e validação:", len(g_tr & g_va),
      "| treino e teste:", len(g_tr & g_te), "| validação e teste:", len(g_va & g_te))

In [ ]:
# Quais gravações aparecem em mais de um conjunto, e com que rótulo?
compartilhadas = (g_tr & g_va) | (g_tr & g_te) | (g_va & g_te)
if compartilhadas:
    display(meta[meta["fsID"].isin(compartilhadas)].groupby(["fsID", "fold", "class"]).size().rename("trechos").to_frame())

# Trechos de cada classe em cada fold
plt.figure(figsize=(10, 4.5))
sns.heatmap(pd.crosstab(meta["class"], meta["fold"]), annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Trechos por classe em cada fold oficial"); plt.xlabel("fold"); plt.ylabel("")
plt.show()

**Uma gravação que escapou da regra:** a conferência acima acha **uma** gravação (`fsID` 180937) no treino e na validação. E com **rótulos diferentes**: os trechos do fold 1 estão como `jackhammer` (britadeira), e os do fold 9, como `drilling` (furadeira). É um **erro da própria base**: a mesma gravação foi rotulada de dois jeitos, e por isso foi parar em dois folds. Não mexemos nela: os folds oficiais são a regra do projeto.

**Leitura:** a validação e o teste têm só ~30 tiros cada. Um ou dois tiros a mais ou a menos mudam o recall do tiro em vários pontos: é um número que **balança muito**, e isso vai para as limitações.

## Etapa 4 — Pré-processamento: do áudio ao espectrograma · Seção 2

A CNN da Aula 4 recebe imagens. Transformamos cada áudio numa imagem:

| Passo | Função | Por quê |
|---|---|---|
| 1. Ler em **mono** e a **22.050 Hz** | `librosa.load(..., sr=22050, mono=True)` | os arquivos chegam com taxas (de 8.000 a 192.000 Hz) e canais diferentes; a rede precisa de tudo no mesmo formato |
| 2. Fixar em **4 s** | ver as duas versões abaixo | todo trecho precisa ter o mesmo tamanho |
| 3. **Espectrograma de mel** | `librosa.feature.melspectrogram` | separa o som em 64 faixas de frequência ao longo do tempo; a escala mel dá mais detalhe aos graves, como o ouvido |
| 4. Em **decibéis**, relativo ao pico | `librosa.power_to_db(..., ref=np.max)` | o log comprime a faixa do som; e o pico de **cada** trecho vira 0 dB, então trechos gravados mais altos ou mais baixos ficam comparáveis |

A célula abaixo confere, numa amostra de 500 arquivos, que as taxas e os canais variam mesmo (só roda se os áudios estiverem baixados).

Cada trecho vira uma matriz de **64 × 173**: 64 faixas de frequência e 173 instantes (88.200 amostras ÷ 512, mais 1).

### As duas versões da entrada

Os trechos curtos precisam ser completados até 4 s. Calculamos **duas versões** de cada trecho, para comparar na Etapa 9:

- **Original:** completa o fim com **silêncio** (zeros). É o que os notebooks públicos fazem.
- **Sem atalho:** coloca o som numa **posição sorteada** dentro dos 4 s e soma um **ruído de fundo fraco** (20 dB abaixo do som) ao trecho inteiro. O ruído vai em **todos** os trechos, curtos e longos: se só os curtos tivessem ruído, "ter ruído" viraria um novo atalho.

A Etapa 5 mostra por que a segunda versão é necessária. Na mesma passada, medimos em cada trecho **onde está o pico do som** e **o volume do pico**, para investigar os atalhos.

**Demora:** cerca de 10 a 15 minutos para os 8.732 trechos. O resultado vai para o Drive. Guardamos também o áudio do fold de teste, para os testes das Etapas 12 e 14.

In [ ]:
if os.path.exists(BASE):
    import soundfile as sf               # vem junto com a librosa; lê só o cabeçalho do arquivo
    amostra = meta.sample(500, random_state=42)
    cabecalhos = [sf.info(os.path.join(BASE, "audio", f"fold{l.fold}", l.slice_file_name)) for l in amostra.itertuples()]
    taxas = pd.Series([c.samplerate for c in cabecalhos])
    canais = pd.Series([c.channels for c in cabecalhos])
    print("Taxas de amostragem (Hz) na amostra:", taxas.value_counts().to_dict())
    print("Número de canais na amostra:", canais.value_counts().to_dict())
else:
    print("Áudios não baixados nesta sessão (os espectrogramas vieram do Drive).")

In [ ]:
SR = 22050                         # taxa de amostragem: 22.050 amostras por segundo
DURACAO = 4.0                      # todo trecho vira 4 s
N_AMOSTRAS = int(SR * DURACAO)     # 88.200 amostras
N_MELS = 64                        # 64 faixas de frequência
HOP = 512                          # de quantas em quantas amostras o espectrograma dá um "passo" no tempo
SNR_FUNDO = 20                     # o ruído de fundo fica 20 dB abaixo do som

def caminho(linha):
    return os.path.join(BASE, "audio", f"fold{linha.fold}", linha.slice_file_name)

def carregar_bruto(arquivo):
    # Lê em mono, na taxa SR, no máximo 4 s. Pode voltar com menos de 4 s.
    y, _ = librosa.load(arquivo, sr=SR, mono=True, duration=DURACAO)
    return y

def versao_original(y):
    # Completa o fim com zeros (silêncio) até 4 s
    return librosa.util.fix_length(y, size=N_AMOSTRAS)

def versao_sem_atalho(y, rng, snr_db=SNR_FUNDO):
    # Coloca o som numa posição sorteada dentro dos 4 s e soma um ruído de fundo fraco
    saida = np.zeros(N_AMOSTRAS, dtype="float32")
    inicio = rng.integers(0, N_AMOSTRAS - len(y) + 1)
    saida[inicio:inicio + len(y)] = y
    potencia_som = np.mean(y ** 2)
    ruido = rng.normal(0, np.sqrt(potencia_som / 10 ** (snr_db / 10)), N_AMOSTRAS)
    return saida + ruido.astype("float32")

def espectrograma(y):
    # Espectrograma de mel em dB: o ponto mais forte do trecho vira 0 dB; o mais fraco, -80 dB
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=2048, hop_length=HOP, n_mels=N_MELS)
    return librosa.power_to_db(S, ref=np.max, top_db=80)

N_QUADROS = 1 + N_AMOSTRAS // HOP   # 173 instantes

if os.path.exists(ARQ_CACHE):
    dados = np.load(ARQ_CACHE)
    XA, XB = dados["XA"], dados["XB"]           # ficam em float16 na memória, para caber no Colab
    ondas_teste, n_teste = dados["ondas_teste"].astype("float32"), dados["n_teste"]
    medidas = dados["medidas"]
    print("Espectrogramas carregados do cache:", XA.shape)
else:
    XA = np.zeros((len(meta), N_MELS, N_QUADROS), dtype="float32")   # versão original
    XB = np.zeros_like(XA)                                            # versão sem atalho
    medidas, ondas_teste, n_teste = [], [], []
    for i, linha in enumerate(tqdm(meta.itertuples(), total=len(meta), desc="Espectrogramas")):
        y = carregar_bruto(caminho(linha))
        rng = np.random.default_rng(i)                 # sorteio repetível, um por trecho
        XA[i] = espectrograma(versao_original(y))
        XB[i] = espectrograma(versao_sem_atalho(y, rng))
        pico = np.abs(y).max()
        medidas.append([np.abs(y).argmax() / SR, 20 * np.log10(pico + 1e-12)])   # onde está o pico e quanto vale
        if linha.fold == 10:
            ondas_teste.append(versao_original(y)); n_teste.append(len(y))
    medidas = np.array(medidas, dtype="float32")
    ondas_teste, n_teste = np.array(ondas_teste, dtype="float32"), np.array(n_teste)
    XA, XB = XA.astype("float16"), XB.astype("float16")   # metade da memória; a precisão sobra para valores em dB
    np.savez(ARQ_CACHE, XA=XA, XB=XB, ondas_teste=ondas_teste.astype("float16"), n_teste=n_teste, medidas=medidas)
    print("Espectrogramas calculados e salvos:", XA.shape)

meta["pico_em_s"], meta["pico_dbfs"] = medidas[:, 0], medidas[:, 1]
print("Cada trecho:", XA.shape[1:], "→", N_MELS, "faixas de frequência ×", N_QUADROS, "instantes")

**Auditoria de vazamento (Aula 5):** além das gravações (`fsID`, Etapa 3), conferimos se há **trechos idênticos** em folds diferentes, por exemplo o mesmo arquivo copiado com outro nome. Comparamos uma "impressão digital" (hash) de cada espectrograma da versão original.

In [ ]:
import hashlib
impressao = pd.Series([hashlib.md5(x.tobytes()).hexdigest() for x in XA])
grupos_iguais = meta.groupby(impressao.values)["fold"].agg(["size", "nunique"])
print("Grupos de trechos idênticos:", int((grupos_iguais["size"] > 1).sum()),
      "| em mais de um fold:", int((grupos_iguais["nunique"] > 1).sum()))

### 4.1 Ver e ouvir: um exemplo de cada classe, e um tiro nas duas versões

O **espectrograma de mel** mostra o som como uma imagem: o eixo horizontal é o tempo, o vertical é a frequência (grave embaixo, agudo em cima), e a cor é a intensidade.

In [ ]:
exemplos = meta[treino].groupby("classID").head(1).sort_values("classID")
fig, eixos = plt.subplots(2, 5, figsize=(16, 5.5))
for eixo, (i, linha) in zip(eixos.flat, exemplos.iterrows()):
    eixo.imshow(XA[i].astype("float32"), origin="lower", aspect="auto", cmap="magma")
    eixo.set_title(f"{linha['class']} ({linha.duracao:.1f} s)", fontsize=10)
    eixo.set_xlabel("tempo"); eixo.set_ylabel("frequência (mel)")
plt.suptitle("Versão original: um exemplo de cada classe"); plt.tight_layout(); plt.show()

i_tiro = exemplos.index[exemplos["classID"] == TIRO][0]
fig, (e1, e2) = plt.subplots(1, 2, figsize=(12, 3.5))
e1.imshow(XA[i_tiro].astype("float32"), origin="lower", aspect="auto", cmap="magma"); e1.set_title("tiro — versão original (silêncio no fim)")
e2.imshow(XB[i_tiro].astype("float32"), origin="lower", aspect="auto", cmap="magma"); e2.set_title("o mesmo tiro — versão sem atalho")
for e in (e1, e2): e.set_xlabel("tempo"); e.set_ylabel("frequência (mel)")
plt.tight_layout(); plt.show()

if os.path.exists(BASE):
    for nome in ["gun_shot", "dog_bark", "siren"]:
        linha = exemplos[exemplos["class"] == nome].iloc[0]
        print(nome, f"({linha.duracao:.1f} s)")
        display(Audio(caminho(linha)))

**Repare:** na versão original, o tiro ocupa o começo e o resto fica **todo escuro** (silêncio digital). Na versão sem atalho, o mesmo tiro aparece em outra posição, cercado de um ruído fraco, como chegaria numa gravação de rua.

### 4.2 Tiros somados a outros sons

Na base, cada trecho tem **um** som. Na rua, o tiro chega junto com trânsito, música, sirene. Para medir isso (e, na Etapa 11.5, para treinar a rede com isso), criamos **misturas**: cada tiro somado a um trecho de **outra classe do mesmo fold**, com o tiro 0, 6 ou 12 dB acima do outro som. Usar o mesmo fold garante que nenhuma gravação cruze entre treino, validação e teste.

São 374 tiros × 3 = 1.122 misturas. Ficam no Drive junto com os espectrogramas. Precisa dos áudios só na primeira vez.

In [ ]:
SNRS_MISTURA = [0, 6, 12]          # o tiro fica 0, 6 ou 12 dB acima do outro som

def misturar(tiro, fundo_bruto, rng, snr_db):
    # O fundo vira 4 s na versão sem atalho (posição sorteada, ruído fraco); o tiro entra numa posição
    # sorteada, com volume ajustado para ficar snr_db acima da potência do fundo
    fundo = versao_sem_atalho(fundo_bruto, rng)
    saida = fundo.copy()
    inicio = rng.integers(0, N_AMOSTRAS - len(tiro) + 1)
    ganho = np.sqrt(np.mean(fundo ** 2) * 10 ** (snr_db / 10) / max(np.mean(tiro ** 2), 1e-12))
    saida[inicio:inicio + len(tiro)] += (ganho * tiro).astype("float32")
    return saida

if os.path.exists(ARQ_MISTURAS):
    dados_m = np.load(ARQ_MISTURAS)
    XM, OM = dados_m["XM"], dados_m["OM"]
    print("Misturas carregadas do cache:", XM.shape)
else:
    rng = np.random.default_rng(2024)
    XM, OM = [], []        # OM: índice do tiro, índice do fundo, snr, classe do fundo, fold
    for fold in range(1, 11):
        tiros = np.where((meta["fold"].values == fold) & (meta["classID"].values == TIRO))[0]
        fundos = np.where((meta["fold"].values == fold) & (meta["classID"].values != TIRO))[0]
        for i in tqdm(tiros, desc=f"Misturas do fold {fold}"):
            y_tiro = carregar_bruto(caminho(meta.iloc[i]))
            for snr in SNRS_MISTURA:
                j = int(rng.choice(fundos))       # o outro som vem do MESMO fold: nada cruza entre conjuntos
                XM.append(espectrograma(misturar(y_tiro, carregar_bruto(caminho(meta.iloc[j])), rng, snr)))
                OM.append([i, j, snr, meta["classID"].values[j], fold])
    XM, OM = np.array(XM, dtype="float16"), np.array(OM)
    np.savez(ARQ_MISTURAS, XM=XM, OM=OM)
    print("Misturas calculadas e salvas:", XM.shape)

**Uma decisão de dados, não de rede:** as misturas usam a mesma preparação da versão sem atalho (posição sorteada, ruído fraco), então não reintroduzem o silêncio nem a posição fixa do pico.

## Etapa 5 — Os atalhos nos dados · Seções 2 e 5

**Shortcut learning** (Aula 2): a rede pode usar qualquer padrão que separe as classes no treino, mesmo que esse padrão **não exista no uso real**. O exemplo da aula é o husky e o lobo, separados pela neve no fundo da foto.

Nesta base, há três candidatos:
1. **Silêncio:** o tiro é curto, então ganha muito silêncio para completar 4 s.
2. **Posição:** quem cortou as gravações começou o trecho exatamente no estampido.
3. **Volume:** tiros costumam ser gravados no volume máximo.

Medimos os três nos folds de treino.

In [ ]:
atalhos = meta[treino].groupby("class").agg(
    duracao_media_s=("duracao", "mean"),
    pct_pico_no_primeiro_quarto_de_segundo=("pico_em_s", lambda s: 100 * (s < 0.25).mean()),
    pico_mediano_dbfs=("pico_dbfs", "median"),
).sort_values("duracao_media_s").round(1)
atalhos

**Quanto cada atalho vale sozinho?** Uma árvore de decisão (disciplina de ML) que recebe **uma única medida**, sem ouvir o som. Treino nos folds 1 a 8, avaliação na validação (fold 9).

In [ ]:
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import f1_score

def metricas(y_real, y_prev):
    # Acurácia geral, F1-macro (média do F1 das 10 classes, cada uma com o mesmo peso),
    # e recall e precisão da classe tiro
    tiro_real, tiro_prev = (y_real == TIRO), (y_prev == TIRO)
    acertos_tiro = (tiro_real & tiro_prev).sum()
    return {"acurácia (%)": round(float(100 * (y_real == y_prev).mean()), 1),
            "F1-macro (%)": round(float(100 * f1_score(y_real, y_prev, average="macro", labels=range(10), zero_division=0)), 1),
            "recall tiro (%)": round(float(100 * acertos_tiro / max(tiro_real.sum(), 1)), 1),
            "precisão tiro (%)": round(float(100 * acertos_tiro / max(tiro_prev.sum(), 1)), 1)}

linhas = []
for coluna, nome in [("duracao", "só a duração (silêncio)"), ("pico_em_s", "só a posição do pico"),
                     ("pico_dbfs", "só o volume do pico")]:
    arvore = DecisionTreeClassifier(max_depth=4, random_state=42)
    arvore.fit(meta.loc[treino, [coluna]], y_treino)
    linhas.append({"atalho": nome, **metricas(y_valid, arvore.predict(meta.loc[valid, [coluna]]))})
tabela_atalhos = pd.DataFrame(linhas).set_index("atalho")
tabela_atalhos

**Leitura:** nenhuma medida sozinha separa bem as 10 classes, mas todas **acham boa parte dos tiros sem ouvir nada**. Uma rede treinada com a versão original pode tirar boa nota no tiro usando esses padrões, e falhar quando eles não existirem.

**O que fazemos:**
1. Treinamos redes com as **duas versões** da entrada e comparamos (Etapas 7 a 10).
2. Avaliamos **todas** as redes numa **validação sem atalho** (o fold 9 na versão sem atalho). É a validação que mais se parece com o uso real, e é por ela que escolhemos o modelo (Etapa 11).
3. O volume já está neutralizado pelo pré-processamento: como o pico de cada trecho vira 0 dB, a rede não vê o volume absoluto. Só a saturação (o som "estourado") continua visível, e ela é em parte real: microfones saturam com um disparo próximo.

## Etapa 6 — Preparar a entrada da rede e o baseline · Seção 3

### 6.1 Padronizar, com estatísticas só do treino

A rede treina melhor com números pequenos, perto de zero (Aula 3). Subtraímos a média e dividimos pelo desvio padrão, **calculados só com o treino** da versão usada para treinar. Usar validação e teste no cálculo seria vazamento (Aula 2).

A função `montar` também acrescenta o canal: uma `Conv2D` espera `(altura, largura, canais)`, como no MNIST (Aula 4). Com o **delta** (Etapa 10), a imagem passa a ter 2 canais; com o **delta do delta** (Etapa 11.1), 3.

In [ ]:
def camadas_de_entrada(e, canais):
    # canal 1: o espectrograma; canal 2: o delta (variação no tempo); canal 3: o delta do delta (aceleração)
    saida = [e]
    if canais >= 2:
        saida.append(librosa.feature.delta(e, axis=-1))
    if canais >= 3:
        saida.append(librosa.feature.delta(e, order=2, axis=-1))
    return saida

def estatisticas(espectros_treino, canais=1):
    e = espectros_treino.astype("float32")
    partes = camadas_de_entrada(e, canais)
    return {"canais": canais, "media": [float(x.mean()) for x in partes], "desvio": [float(x.std()) for x in partes]}

def montar(espectros, s, bloco=1000):
    # padroniza cada canal com as estatísticas DO TREINO e empilha os canais.
    # Faz de 1.000 em 1.000 trechos, para não estourar a memória do Colab.
    saida = np.empty((len(espectros), N_MELS, N_QUADROS, s["canais"]), dtype="float32")
    for i in range(0, len(espectros), bloco):
        partes = camadas_de_entrada(espectros[i:i + bloco].astype("float32"), s["canais"])
        for c, (x, m, d) in enumerate(zip(partes, s["media"], s["desvio"])):
            saida[i:i + bloco, ..., c] = (x - m) / d
    return saida

stats_A = estatisticas(XA[treino])          # para as redes treinadas com a versão original
stats_B = estatisticas(XB[treino])          # para as redes treinadas com a versão sem atalho
print("Versão original — média:", round(stats_A["media"][0], 1), "dB | desvio:", round(stats_A["desvio"][0], 1), "dB")
print("Versão sem atalho — média:", round(stats_B["media"][0], 1), "dB | desvio:", round(stats_B["desvio"][0], 1), "dB")

### 6.2 Baseline: o modelo sem aprendizado

Duas referências:
1. **Prever sempre a classe mais frequente do treino**, sem olhar o áudio (a pedida na descrição do projeto). Com 10 classes quase equilibradas, acerta ≈ 11% e nunca acerta um tiro.
2. **Random Forest** (disciplina de ML) com um resumo simples do espectrograma: a média e o desvio de cada faixa de frequência (128 números por trecho). Mostra quanto um modelo **sem deep learning** consegue. Se a CNN não passar bem dele, a rede não se justifica.

**Métricas:** a principal é a **acurácia**. Ao lado, o **F1-macro** (a média do F1 das 10 classes, cada classe com o mesmo peso: uma classe pequena, como o tiro, pesa tanto quanto as grandes) e o recall e a precisão do tiro.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(y_treino), 1)), y_treino)
prev_base_valid = baseline.predict(np.zeros((len(y_valid), 1)))
m_base = metricas(y_valid, prev_base_valid)
print("Classe que o baseline sempre prevê:", CLASSES[prev_base_valid[0]])
print("Baseline (validação):", m_base)

# Baseline 2: Random Forest (disciplina de ML) com 128 números por trecho: a média e o desvio de cada
# uma das 64 faixas de mel ao longo do tempo. Treinado na versão sem atalho, como as CNNs 3 em diante.
from sklearn.ensemble import RandomForestClassifier

def resumo_faixas(espectros):
    e = espectros.astype("float32")
    return np.concatenate([e.mean(axis=2), e.std(axis=2)], axis=1)     # (trechos, 128)

floresta = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
floresta.fit(resumo_faixas(XB[treino]), y_treino)
m_rf_orig = metricas(y_valid, floresta.predict(resumo_faixas(XA[valid])))
m_rf = metricas(y_valid, floresta.predict(resumo_faixas(XB[valid])))
print("Random Forest (validação sem atalho):", m_rf)

# Os resultados de validação de todos os modelos vão para esta lista
resultados = [{"modelo": "Baseline (classe mais frequente)", "treinado com": "—",
               "acurácia valid. original (%)": m_base["acurácia (%)"],
               "acurácia valid. sem atalho (%)": m_base["acurácia (%)"],
               "F1-macro valid. sem atalho (%)": m_base["F1-macro (%)"],
               "recall tiro sem atalho (%)": 0.0, "precisão tiro sem atalho (%)": 0.0,
               "o que é": "referência sem aprendizado"},
              {"modelo": "Baseline (Random Forest)", "treinado com": "sem atalho",
               "acurácia valid. original (%)": m_rf_orig["acurácia (%)"],
               "acurácia valid. sem atalho (%)": m_rf["acurácia (%)"],
               "F1-macro valid. sem atalho (%)": m_rf["F1-macro (%)"],
               "recall tiro sem atalho (%)": m_rf["recall tiro (%)"], "precisão tiro sem atalho (%)": m_rf["precisão tiro (%)"],
               "o que é": "ML tradicional: média e desvio de cada faixa de mel"}]

## Etapa 7 — Primeira rede: uma CNN pequena · Seções 3 e 4

A mesma ideia da CNN da Aula 4: blocos de `Conv2D` (filtros que percorrem a imagem) e `MaxPooling2D` (fica com o maior valor de cada bloco 2 × 2), depois `Flatten` e a camada de saída.

**Diferença para o MNIST:** o espectrograma (64 × 173) é bem maior que um dígito (28 × 28). Com dois blocos, sobrariam 36.736 números no `Flatten` e só a camada de saída teria 367 mil parâmetros. Com **três blocos**, sobram 7.296, e a rede inteira fica com ≈ 129 mil parâmetros.

| Camada | Tamanho da saída | Parâmetros |
|---|---|---|
| entrada | 64 × 173 × 1 | 0 |
| `Conv2D(32, 3)` + ReLU | 62 × 171 × 32 | (3·3·1 + 1) × 32 = 320 |
| `MaxPooling2D(2)` | 31 × 85 × 32 | 0 |
| `Conv2D(64, 3)` + ReLU | 29 × 83 × 64 | (3·3·32 + 1) × 64 = 18.496 |
| `MaxPooling2D(2)` | 14 × 41 × 64 | 0 |
| `Conv2D(64, 3)` + ReLU | 12 × 39 × 64 | (3·3·64 + 1) × 64 = 36.928 |
| `MaxPooling2D(2)` | 6 × 19 × 64 | 0 |
| `Flatten` | 7.296 | 0 |
| `Dense(10)` + softmax | 10 | (7.296 + 1) × 10 = 72.970 |
| **total** | | **128.714** |

**A CNN 1 treina com a versão original** (com silêncio), como os notebooks públicos.

In [ ]:
@keras.saving.register_keras_serializable()          # permite carregar o modelo salvo (Etapa 14)
class MascaraTempoFreq(layers.Layer):
    # Aumento de dados (SpecAugment simplificado): durante o treino, apaga uma faixa de frequência
    # (até max_f faixas de mel) e um trecho de tempo (até max_t quadros) de cada exemplo, sorteados.
    # O valor 0 é a média, porque a entrada já está padronizada. Fora do treino, não faz nada.
    def __init__(self, max_f=8, max_t=20, semente=0, **kwargs):
        super().__init__(**kwargs)
        self.max_f, self.max_t, self.semente = max_f, max_t, semente
        self.gerador = keras.random.SeedGenerator(semente)

    def get_config(self):                 # o que o Keras guarda para recriar a camada ao carregar
        return {**super().get_config(), "max_f": self.max_f, "max_t": self.max_t, "semente": self.semente}

    def call(self, x, training=None):
        if not training:
            return x
        ops, n = keras.ops, keras.ops.shape(x)[0]
        altura, largura = x.shape[1], x.shape[2]
        sorteio = lambda: keras.random.uniform((n, 1, 1, 1), seed=self.gerador)
        f = ops.floor(sorteio() * (self.max_f + 1)); f0 = ops.floor(sorteio() * (altura - f + 1))
        t = ops.floor(sorteio() * (self.max_t + 1)); t0 = ops.floor(sorteio() * (largura - t + 1))
        linha = ops.reshape(ops.arange(altura, dtype="float32"), (1, altura, 1, 1))
        coluna = ops.reshape(ops.arange(largura, dtype="float32"), (1, 1, largura, 1))
        apaga = ((linha >= f0) & (linha < f0 + f)) | ((coluna >= t0) & (coluna < t0 + t))
        return x * ops.cast(ops.logical_not(apaga), x.dtype)

def criar_cnn(nome, dropout=0.0, canais=1, l2=0.0, aumento=False, desloc_freq=0.0, mascaras=False, semente=42):
    keras.utils.set_random_seed(semente)  # mesmos pesos iniciais em todos os experimentos
    reg = keras.regularizers.l2(l2) if l2 > 0 else None        # regularização L2 (Etapa 11.1)
    camadas = [keras.Input(shape=(N_MELS, N_QUADROS, canais))]
    if aumento:                                                # aumento de dados (Aula 4), só no treino
        camadas.append(layers.RandomTranslation(height_factor=0.0, width_factor=0.1, fill_mode="wrap"))
    if desloc_freq > 0:                                        # aumento de dados na frequência (Etapa 11.3)
        camadas.append(layers.RandomTranslation(height_factor=desloc_freq, width_factor=0.0, fill_mode="constant"))
    if mascaras:                                               # apaga faixas de tempo e frequência (Etapa 11.4)
        camadas.append(MascaraTempoFreq(semente=semente))
    for filtros in (32, 64, 64):
        camadas += [layers.Conv2D(filtros, 3, activation="relu", kernel_regularizer=reg), layers.MaxPooling2D(2)]
        if dropout > 0:
            camadas.append(layers.Dropout(dropout))
    camadas.append(layers.Flatten())
    if dropout > 0:
        camadas.append(layers.Dropout(dropout))
    camadas.append(layers.Dense(10, activation="softmax", kernel_regularizer=reg))   # 10 probabilidades
    return keras.Sequential(camadas, name=nome)

cnn1 = criar_cnn("cnn1")
cnn1.summary()

### 7.1 Treinar

A receita da Aula 3, igual para todos os experimentos:

| Escolha | Valor | Por quê |
|---|---|---|
| Otimizador | Adam, taxa 0,001 | o ponto de partida da Aula 3 |
| Perda | `sparse_categorical_crossentropy` | multiclasse com rótulos inteiros (0 a 9) |
| Métrica | `accuracy` | para nós lermos; o recall do tiro é calculado à parte |
| Lote | 32 | o padrão do Keras: 7.079 ÷ 32 ≈ 222 atualizações por época |
| Épocas | até 60, com `EarlyStopping` | para quando a **acurácia** de validação não melhora há 8 épocas, e volta aos pesos da melhor época |

**Por que monitorar a acurácia, e não a perda, como na Aula 3?** Na primeira versão deste notebook, o `EarlyStopping` olhava a perda de validação (`val_loss`). Ela parava de cair já na época 3 ou 4 e depois subia, enquanto a **acurácia** de validação continuava melhorando até a época 9 a 11. A perda sobe porque, nos trechos que erra, a rede fica cada vez mais segura do erro, e a entropia cruzada pune muito o erro com certeza (Aula 3). Com `val_loss`, todas as redes voltavam a pesos de uma época muito cedo: a CNN 2, por exemplo, voltava à época 4, com 61,5% de acurácia, embora tivesse chegado a 72,7%. Como a métrica que nos importa é a acurácia, passamos a monitorá-la. É uma escolha feita só com a validação.

**Taxa e lote** ficam fixos até a Etapa 10; a rodada 2 (Etapa 11.3) testa outros valores.

**Por que paciência 8, e não 5 como na aula?** Com dropout, a validação oscila de uma época para outra; com 5, o treino poderia parar cedo por causa de uma oscilação. É a recomendação da Aula 5 (slide 14): quando a validação oscila, uma paciência maior espera pela próxima melhora. Usamos 8 em todos os experimentos, para compará-los do mesmo jeito.

**Cada rede usa a validação da sua própria versão** para o `EarlyStopping`: as treinadas com a versão original olham o fold 9 original; as treinadas sem atalho, o fold 9 sem atalho.

**Cache:** a rede treinada fica salva. Com `RETREINAR = False`, rodar de novo só carrega os pesos e o histórico.

In [ ]:
RETREINAR = False
VERSAO_TREINO = "acc"     # muda o nome dos arquivos salvos quando a receita de treino muda

def treinar(modelo, X_tr, X_va, reduzir_lr=False, taxa=0.001, lote=32, y_tr=None, y_va=None):
    y_tr = y_treino if y_tr is None else y_tr          # a validação cruzada (13.6) passa outros rótulos
    y_va = y_valid if y_va is None else y_va
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=taxa),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    arq_pesos = os.path.join(PASTA_MODELOS, f"{modelo.name}_{VERSAO_TREINO}.weights.h5")
    arq_hist = os.path.join(PASTA_MODELOS, f"{modelo.name}_{VERSAO_TREINO}_historico.json")
    if os.path.exists(arq_pesos) and os.path.exists(arq_hist) and not RETREINAR:
        modelo.load_weights(arq_pesos)
        historico = json.load(open(arq_hist))
        print(f"{modelo.name}: pesos e histórico carregados do cache ({len(historico['loss'])} épocas)")
        return historico
    chamadas = [keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=8,
                                              restore_best_weights=True)]
    if reduzir_lr == "acuracia":   # Etapa 11.4: taxa × 0,3 quando a acurácia de validação estaciona por 3 épocas
        chamadas.append(keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", mode="max", factor=0.3,
                                                          patience=3, min_lr=1e-5))
    elif reduzir_lr:               # Aula 3: divide a taxa por 2 quando a perda de validação estaciona por 3 épocas
        chamadas.append(keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3))
    hist = modelo.fit(X_tr, y_tr, validation_data=(X_va, y_va),
                      epochs=60, batch_size=lote, callbacks=chamadas, verbose=2)
    historico = {k: [float(v) for v in vals] for k, vals in hist.history.items() if k != "learning_rate"}
    modelo.save_weights(arq_pesos)
    json.dump(historico, open(arq_hist, "w"))
    return historico

def desenhar_curvas(h, titulo):
    epocas = range(1, len(h["loss"]) + 1)
    melhor = int(np.argmax(h["val_accuracy"])) + 1
    fig, (e1, e2) = plt.subplots(1, 2, figsize=(12, 3.8))
    e1.plot(epocas, h["loss"], "o-", ms=3, label="treino")
    e1.plot(epocas, h["val_loss"], "o-", ms=3, label="validação")
    e1.set_title("Perda (menor é melhor)")
    e2.plot(epocas, h["accuracy"], "o-", ms=3, label="treino")
    e2.plot(epocas, h["val_accuracy"], "o-", ms=3, label="validação")
    e2.set_title("Acurácia (maior é melhor)")
    for e in (e1, e2):
        e.axvline(melhor, ls=":", color="gray", label=f"melhor época ({melhor})")
        e.set_xlabel("época"); e.legend()
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

def prever(modelo, X, bloco=2048):
    # prevê em pedaços, para não copiar o conjunto inteiro de uma vez para o TensorFlow (economiza memória)
    return np.concatenate([modelo.predict(X[i:i + bloco], batch_size=256, verbose=0) for i in range(0, len(X), bloco)])

def registrar(nome, modelo, historico, X_tr, s, versao, mudanca):
    # Avalia a rede no treino, na validação original e na validação sem atalho
    acc_treino = 100 * (prever(modelo, X_tr).argmax(1) == y_treino).mean()
    p_orig = prever(modelo, montar(XA[valid], s)).argmax(1)
    p_sem = prever(modelo, montar(XB[valid], s)).argmax(1)
    m_orig, m_sem = metricas(y_valid, p_orig), metricas(y_valid, p_sem)
    linha = {"modelo": nome, "treinado com": versao,
             "acurácia treino (%)": round(float(acc_treino), 1),
             "acurácia valid. original (%)": m_orig["acurácia (%)"],
             "acurácia valid. sem atalho (%)": m_sem["acurácia (%)"],
             "F1-macro valid. sem atalho (%)": m_sem["F1-macro (%)"],
             "recall tiro original (%)": m_orig["recall tiro (%)"],
             "recall tiro sem atalho (%)": m_sem["recall tiro (%)"],
             "precisão tiro sem atalho (%)": m_sem["precisão tiro (%)"],
             "melhor época": int(np.argmax(historico["val_accuracy"])) + 1,
             "parâmetros": modelo.count_params(), "o que é": mudanca}
    resultados.append(linha)
    return pd.Series(linha)

X_tr_A, X_va_A = montar(XA[treino], stats_A), montar(XA[valid], stats_A)
hist1 = treinar(cnn1, X_tr_A, X_va_A)
desenhar_curvas(hist1, "CNN 1 — versão original, sem regularização")
registrar("CNN 1", cnn1, hist1, X_tr_A, stats_A, "original", "primeira rede, sem regularização")

**Como ler as curvas (Aula 3, "leia a curva"):**
- As duas perdas caem juntas no início: a rede está aprendendo.
- Se a **perda de treino continua caindo e a de validação para ou sobe**, é **sobreajuste**: a rede decora o treino. Aqui a perda de validação sobe cedo, mas a acurácia de validação ainda melhora por algumas épocas: a rede acerta mais trechos, mas erra os outros com mais certeza. A distância entre a acurácia de treino e a de validação, impressa acima, mede isso.
- Se treino e validação ficassem **altos e juntos**, seria viés: a rede pequena demais.

**Compare as duas colunas de validação:** a mesma rede, no mesmo fold 9, com e sem os atalhos. Se a acurácia e o recall do tiro caírem muito na versão sem atalho, a rede dependia deles.

## Etapa 8 — CNN 2: regularização com dropout · Seção 4

**O que muda:** só uma coisa, camadas `Dropout(0.3)` depois de cada bloco e antes da saída. Durante o treino, o dropout desliga, a cada lote, 30% das saídas da camada anterior, sorteadas. A rede não pode depender de nenhum neurônio específico e precisa espalhar o que aprende. No `evaluate` e no `predict`, nada é desligado. O dropout não tem pesos: o número de parâmetros não muda.

**Por que este experimento:** se a CNN 1 sobreajustar (treino bem acima da validação), a regularização é o remédio da Aula 2.

**O que esperamos:** treino um pouco pior, validação igual ou melhor, curvas mais próximas.

In [ ]:
cnn2 = criar_cnn("cnn2_dropout", dropout=0.3)
hist2 = treinar(cnn2, X_tr_A, X_va_A)
desenhar_curvas(hist2, "CNN 2 — versão original, com dropout")
registrar("CNN 2 (dropout)", cnn2, hist2, X_tr_A, stats_A, "original", "CNN 1 + Dropout(0.3)")

## Etapa 9 — CNN 3: a mesma rede, treinada sem os atalhos · Seção 4

**O que muda:** só a entrada. A arquitetura é a da CNN 2, mas ela treina com a **versão sem atalho** (som em posição sorteada, ruído fraco no lugar do silêncio).

**Por que este experimento:** a Etapa 5 mostrou que silêncio e posição separam os tiros sem ouvir o som. Se a rede não puder contar com eles, precisa aprender o som do tiro.

**O que esperamos:** na validação original, talvez um pouco pior que a CNN 2 (ela perde o atalho que a validação original ainda tem). Na validação sem atalho, bem melhor que a CNN 2.

In [ ]:
import gc
del X_tr_A, X_va_A; gc.collect()         # libera memória
X_tr_B, X_va_B = montar(XB[treino], stats_B), montar(XB[valid], stats_B)
cnn3 = criar_cnn("cnn3_sem_atalho", dropout=0.3)
hist3 = treinar(cnn3, X_tr_B, X_va_B)
desenhar_curvas(hist3, "CNN 3 — versão sem atalho, com dropout")
registrar("CNN 3 (sem atalho)", cnn3, hist3, X_tr_B, stats_B, "sem atalho", "CNN 2 treinada sem os atalhos")

## Etapa 10 — CNN 4: acrescentar a variação no tempo (delta) · Seção 4

**O que muda:** a entrada ganha um segundo canal, o **delta**: quanto cada faixa de frequência mudou de um instante para o outro (`librosa.feature.delta`). A imagem passa a ter 2 canais, como uma foto com duas cores. A primeira `Conv2D` ganha pesos para o canal novo; o resto da rede é igual à CNN 3.

**Por que este experimento:** é o equivalente, neste projeto, das colunas criadas no projeto de ML: entregar à rede uma informação derivada. O tiro é uma **mudança brusca**; no delta, um estampido aparece como um salto forte.

**O que esperamos:** ganho pequeno, se houver. A rede já poderia aprender essas diferenças sozinha (Aula 1: no deep learning, as representações são aprendidas). Se não ajudar, é um resultado válido.

In [ ]:
del X_tr_B, X_va_B; gc.collect()
stats_Bd = estatisticas(XB[treino], canais=2)
X_tr_Bd, X_va_Bd = montar(XB[treino], stats_Bd), montar(XB[valid], stats_Bd)
print("Entrada com delta:", X_tr_Bd.shape)
cnn4 = criar_cnn("cnn4_delta", dropout=0.3, canais=2)
hist4 = treinar(cnn4, X_tr_Bd, X_va_Bd)
desenhar_curvas(hist4, "CNN 4 — versão sem atalho, com delta")
display(registrar("CNN 4 (sem atalho + delta)", cnn4, hist4, X_tr_Bd, stats_Bd, "sem atalho", "CNN 3 + canal de delta"))
del X_tr_Bd, X_va_Bd; gc.collect()

# Guardamos como cada rede foi montada, para as etapas seguintes
modelos = {"CNN 1": cnn1, "CNN 2 (dropout)": cnn2, "CNN 3 (sem atalho)": cnn3, "CNN 4 (sem atalho + delta)": cnn4}
stats_de = {"CNN 1": stats_A, "CNN 2 (dropout)": stats_A, "CNN 3 (sem atalho)": stats_B, "CNN 4 (sem atalho + delta)": stats_Bd}
historicos = {"CNN 1": hist1, "CNN 2 (dropout)": hist2, "CNN 3 (sem atalho)": hist3, "CNN 4 (sem atalho + delta)": hist4}
receitas = {"CNN 3 (sem atalho)": dict(canais=1, dropout=0.3, l2=0.0, aumento=False, reduzir_lr=False),
            "CNN 4 (sem atalho + delta)": dict(canais=2, dropout=0.3, l2=0.0, aumento=False, reduzir_lr=False)}

## Etapa 11 — Melhorar a CNN 4, uma rodada de cada vez · Seções 3 e 4

A CNN 4 virou o ponto de partida. Daqui em diante, cada **rodada** testa algumas mudanças, **uma de cada vez**, em cima da melhor rede até ali. A vencedora de cada rodada ganha o próximo número:

| Rodada | Parte de | Vencedora vira | O que se compara |
|---|---|---|---|
| 1 | CNN 4 | **CNN 5** | 5 mudanças, uma semente (triagem); a vencedora é repetida com 3 sementes |
| 2 | CNN 5 | **CNN 6** | 3 mudanças, 3 sementes cada, pela média |
| 3 | CNN 6 | **CNN 7** | 3 mudanças, 3 sementes cada, pela média |
| 4 | CNN 7 | **CNN 8** | 2 mudanças, 3 sementes, com o critério do tiro com outro som |
| 5 | CNN 8 | **CNN 9** (final) | 3 mudanças de aumento de dados e a combinação das que passaram, 3 sementes |
| 6 | CNN 9 | — | transferência de aprendizado com a YAMNet (rede pré-treinada em áudio), 3 sementes |
| 7 | CNN 9 | — | RawBoost no lugar dos microfones simulados, 3 sementes |
| 8 | CNN 9 | — | RawBoost junto com os microfones simulados, 3 sementes |

As mudanças que perdem a sua rodada aparecem só na tabela da rodada: elas também são resultado (mostram o que **não** funciona e por quê).

**Por que 3 sementes a partir da rodada 2?** Os pesos começam sorteados (Aula 4). A mesma rede, só trocando a semente, varia alguns pontos na validação (Etapa 11.2). Diferenças menores que isso não provam nada.

**Regra das rodadas 1 a 3, escrita antes de ver os números:** a mudança vence se passar a referência por **mais de 1 ponto** de acurácia na validação sem atalho (na rodada 1, com uma semente; nas outras, pela média das 3). Se mais de uma passar, fica a de maior acurácia. Se nenhuma passar, a referência continua.

**As técnicas, em uma frase cada:**

| Técnica | O que é | Aula |
|---|---|---|
| Delta do delta | 3º canal da entrada: a "aceleração", quanto o delta mudou de um instante para o outro | — |
| Aumento no tempo | `RandomTranslation` horizontal: a cada época, o espectrograma é deslocado no tempo | 4 |
| Deslocamento na frequência | `RandomTranslation` vertical: o som fica um pouco mais grave ou mais agudo | 4 |
| Dropout 0,5 e L2 | regularização: desligar mais neurônios no treino; multar pesos grandes na perda | 2 |
| Taxa adaptativa | `ReduceLROnPlateau`: diminui a taxa de aprendizado quando a validação estaciona | 3 |
| Taxa 0,0003 e lote 16 | passos menores; mais atualizações por época | 3 |
| Máscaras de tempo e frequência | aumento de dados próprio de espectrogramas (*SpecAugment*): apaga uma faixa de frequência e um trecho de tempo, sorteados a cada lote, só no treino | — |
| Tiros sobre outros sons | aumento de dados da classe de interesse: as misturas da Etapa 4.2, dos folds de treino, entram no treino rotuladas como tiro | 4 |
| Misturas de sons sem tiro | aumento de dados: dois sons que não são tiro somados, com o rótulo do mais forte | 4 |
| Bordas suavizadas | aumento de dados: o começo e o fim de trechos curtos sobem e descem aos poucos | 4 |
| Microfones simulados | aumento de dados: parte do treino passa por um filtro que corta graves e agudos, como um microfone pior | 4 e 5 |
| RawBoost | aumento de dados na onda (Tak e outros, 2022): ruído de canal, estalos e ruído de fundo sorteados | — |
| YAMNet congelada | transferência de aprendizado: uma rede já treinada em 2 milhões de áudios transforma cada trecho em 2.048 números, e só a cabeça é treinada | 4 |

**A regra da aumentação (Aula 5, slide 35):** só vale uma variação que (1) poderia aparecer nos dados reais e (2) não muda a resposta certa. E, pelo slide 42, primeiro se divide, depois se aumenta, e só o treino.

| Aumentação | Poderia aparecer na rua? | Mantém a resposta? |
|---|---|---|
| deslocamento no tempo | sim: o som começa em qualquer instante | sim |
| deslocamento na frequência | em parte: o mesmo motor pode soar mais grave | nem sempre: o tom é parte da identidade de sirene e buzina (e não ajudou) |
| máscaras de tempo e frequência | sim: partes do som encobertas por outros sons | sim |
| tiros somados a outros sons | sim: é o caso de uso | sim: continua sendo tiro. As misturas usam sons do **mesmo fold** e, no treino, só dos folds de treino |
| dois sons sem tiro somados (rodada 5) | sim: a rua tem vários sons ao mesmo tempo | sim, com o rótulo do som mais forte (6 ou 12 dB acima) |
| bordas suavizadas (rodada 5) | sim: um som pode começar e terminar aos poucos | sim |
| microfone simulado (rodada 5) | sim: sensores e celulares cortam graves e agudos | sim: o som continua o mesmo, só mais abafado |

**Tentativas do protótipo deixadas fora do notebook** (nenhuma superou a referência): 128 faixas de mel, `GlobalAveragePooling2D` no lugar do `Flatten`, transferência de aprendizado com a MobileNetV2 treinada no ImageNet (piorou: 74,4% na validação, média de 3 sementes, contra 78,3% da CNN 5, com 17 vezes mais parâmetros; como avisa a Aula 5, slide 43, em outra "modalidade" de imagem a base pré-treinada pode não ajudar, e um espectrograma não é uma foto), e um canal de "pulsação" (quanto a energia oscila dentro de cada instante), que aproximou furadeira de tiro em vez de separar.

**Demora:** cerca de 45 redes, contando as sementes: 1 a 1,5 h com GPU na primeira vez, mais uns 30 min para preparar os dados das rodadas 5 e 6; depois, tudo vem do cache.

In [ ]:
SEMENTES = (42, 7, 123)
infos = {}                                   # medidas de cada receita, uma por semente
stats_por_canais = {1: stats_B, 2: stats_Bd}

def dados_de_treino(r, mascara, s):
    # Monta o treino da receita r com os trechos da máscara (folds de treino) e os aumentos de dados que ela pede
    # (junta os espectrogramas em float16 e só depois padroniza: economiza memória)
    folds = np.unique(meta.loc[mascara, "fold"])
    esp = (R5["XF"] if r.get("bordas") else XB)[mascara]           # rodada 5: bordas suavizadas (Etapa 11.6)
    if r.get("microfones"):                                         # rodada 5: microfones simulados (Etapa 11.6)
        esp = esp.copy(); trocar = R5["CMIC"][mascara]; esp[trocar] = R5["XC"][mascara][trocar]
    if r.get("rawboost"):                   # rodadas 7 e 8: RawBoost no lugar dos microfones (R7) ou junto (R8)
        RB = R7 if r.get("rawboost_onde", "no_lugar") == "no_lugar" else R8
        indices = np.where(mascara)[0]; tem = np.isin(indices, RB["IDX"])
        linhas_rb = np.searchsorted(RB["IDX"], indices[tem])
        esp = esp.copy(); esp[tem] = (RB["XR4"] if r["rawboost"] == "completo" else RB["XR6"])[linhas_rb]
    partes, rotulos = [esp], [meta.loc[mascara, "classID"].values]
    if r.get("misturas") is not None:        # rodada 4: tiros somados a outros sons (só dos folds de treino)
        usar = np.isin(OM[:, 4], folds) & (OM[:, 2] >= r["misturas"])
        partes.append(XM[usar]); rotulos.append(np.full(int(usar.sum()), TIRO))
    if r.get("misturas_sem_tiro"):           # rodada 5: dois sons sem tiro somados, rótulo do mais forte
        usar = np.isin(R5["ON"][:, 5], folds)
        partes.append(R5["XN"][usar]); rotulos.append(R5["ON"][usar, 3])
    return montar(np.concatenate(partes), s), np.concatenate(rotulos)

def medidas_extras(modelo, s):
    # Só a partir da rodada 5 (quando os dados da Etapa 11.6 existem): duas medidas novas, na validação
    if "R5" not in globals():
        return {}
    ON, VT = R5["ON"], R5["VT"]; no9 = meta["fold"].values[VT] == 9
    p_mist = prever(modelo, montar(R5["XN"][ON[:, 5] == 9], s)).argmax(1)
    p_mic = [prever(modelo, montar(R5[k][no9], s)).argmax(1) for k in ("XTEL", "X8K")]
    return {"misturas sem tiro → tiro (%)": round(float(100 * (p_mist == TIRO).mean()), 1),
            "acurácia c/ microfone simulado (%)": round(float(np.mean([100 * (p == y_valid).mean() for p in p_mic])), 1),
            "precisão tiro c/ microfone simulado (%)": round(float(np.mean([100 * (y_valid[p == TIRO] == TIRO).mean() if (p == TIRO).any() else 0 for p in p_mic])), 1)}

def rodar_receita(nome, r, semente=42, registrar_resultado=False, descricao=""):
    # Treina (ou carrega do cache) a receita r com uma semente e devolve as medidas na validação sem atalho
    if r["canais"] not in stats_por_canais:
        stats_por_canais[r["canais"]] = estatisticas(XB[treino], canais=r["canais"])
    s = stats_por_canais[r["canais"]]
    X_tr, y_tr = dados_de_treino(r, treino, s)
    X_va = montar(XB[valid], s)
    arquivo = r["arquivo"] + ("" if semente == 42 else f"_s{semente}")      # nome dos arquivos no cache
    modelo = criar_cnn(arquivo, dropout=r["dropout"], canais=r["canais"], l2=r["l2"], aumento=r["aumento"],
                       desloc_freq=r.get("desloc_freq", 0.0), mascaras=r.get("mascaras", False), semente=semente)
    h = treinar(modelo, X_tr, X_va, reduzir_lr=r["reduzir_lr"], taxa=r.get("taxa", 0.001), lote=r.get("lote", 32), y_tr=y_tr)
    m = metricas(y_valid, prever(modelo, X_va).argmax(1))
    com_fundo = prever(modelo, montar(XM[OM[:, 4] == 9], s)).argmax(1) == TIRO      # misturas do fold 9
    info = {"acurácia valid. sem atalho (%)": m["acurácia (%)"], "F1-macro (%)": m["F1-macro (%)"],
            "recall tiro (%)": m["recall tiro (%)"], "precisão tiro (%)": m["precisão tiro (%)"],
            "tiro com outro som (%)": round(float(100 * com_fundo.mean()), 1),
            "acurácia treino (%)": round(float(100 * (prever(modelo, X_tr).argmax(1) == y_tr).mean()), 1),
            **medidas_extras(modelo, s),
            "melhor época": int(np.argmax(h["val_accuracy"])) + 1, "parâmetros": modelo.count_params()}
    if registrar_resultado:
        registrar(nome, modelo, h, X_tr, s, "sem atalho", descricao)
    del X_tr, X_va; gc.collect()
    return modelo, s, h, info

def medir_sementes(nome, r):
    # Treina a receita com as 3 sementes e guarda a rede da semente 42 para as etapas seguintes
    lista = []
    for semente in SEMENTES:
        modelo, s, h, info = rodar_receita(nome, r, semente=semente)
        lista.append(info)
        if semente == 42:
            modelos[nome], stats_de[nome], historicos[nome], receitas[nome] = modelo, s, h, r
    infos[nome] = lista

def linha_rodada(nome, ref_media=None):
    lista = infos[nome]
    accs = [i["acurácia valid. sem atalho (%)"] for i in lista]
    media = lambda chave: round(float(np.mean([i[chave] for i in lista])), 1)
    return {"rede": nome, "sementes 42 / 7 / 123 (%)": " / ".join(f"{a:.1f}" for a in accs),
            "média (%)": round(float(np.mean(accs)), 1),
            "diferença p/ referência (pontos)": 0.0 if ref_media is None else round(float(np.mean(accs)) - ref_media, 1),
            "F1-macro (%)": media("F1-macro (%)"), "recall tiro (%)": media("recall tiro (%)"),
            "precisão tiro (%)": media("precisão tiro (%)"), "tiro com outro som (%)": media("tiro com outro som (%)"),
            "acurácia treino (%)": media("acurácia treino (%)"),
            **{k: media(k) for k in ("misturas sem tiro → tiro (%)", "acurácia c/ microfone simulado (%)",
                                     "precisão tiro c/ microfone simulado (%)") if k in lista[0]},
            "melhores épocas": " / ".join(str(i["melhor época"]) for i in lista), "parâmetros": lista[0]["parâmetros"]}

def promover(vencedora, numero, ideia):
    # A vencedora de uma rodada ganha o próximo número de CNN (os mesmos pesos e medidas, com outro nome)
    nome = f"CNN {numero} (+ {ideia})"
    modelos[nome], stats_de[nome], historicos[nome] = modelos[vencedora], stats_de[vencedora], historicos[vencedora]
    receitas[nome], infos[nome] = receitas[vencedora], infos.get(vencedora)
    return nome

def rodada(ref, variacoes):
    # Treina as variações com 3 sementes e monta a tabela, com a referência na primeira linha
    for nome, r in variacoes.items():
        medir_sementes(nome, r)
    media_ref = float(np.mean([i["acurácia valid. sem atalho (%)"] for i in infos[ref]]))
    return pd.DataFrame([linha_rodada(ref)] + [linha_rodada(n, media_ref) for n in variacoes]).set_index("rede")

def vencedora_por_acuracia(tabela, ref, variacoes):
    passaram = tabela.loc[list(variacoes)]
    passaram = passaram[passaram["diferença p/ referência (pontos)"] > 1.0]
    return passaram["média (%)"].idxmax() if len(passaram) else None

### 11.1 Rodada 1: cinco mudanças em cima da CNN 4 (triagem, uma semente)

In [ ]:
r4 = {**receitas["CNN 4 (sem atalho + delta)"], "arquivo": "cnn4_delta"}
receitas["CNN 4 (sem atalho + delta)"] = r4
variacoes_r1 = {
    "R1 · delta do delta":     {**r4, "canais": 3, "arquivo": "cnn_cnn4adeltadodelta"},
    "R1 · aumento no tempo":   {**r4, "aumento": True, "arquivo": "cnn_cnn4baumentodedados"},
    "R1 · dropout 0,5":        {**r4, "dropout": 0.5, "arquivo": "cnn_cnn4cdropout05"},
    "R1 · L2":                 {**r4, "l2": 1e-4, "arquivo": "cnn_cnn4dl2"},
    "R1 · taxa adaptativa":    {**r4, "reduzir_lr": True, "arquivo": "cnn_cnn4etaxaadaptativa"},
}
for nome, r in variacoes_r1.items():
    modelo, s, h, _ = rodar_receita(nome, r, registrar_resultado=True, descricao=nome.split("· ")[1])
    modelos[nome], stats_de[nome], historicos[nome], receitas[nome] = modelo, s, h, r

# As que passaram a CNN 4 por mais de 1 ponto entram juntas numa combinação
tabela_valid = pd.DataFrame(resultados).set_index("modelo")
acc4 = tabela_valid.loc["CNN 4 (sem atalho + delta)", "acurácia valid. sem atalho (%)"]
ajudaram = [n for n in variacoes_r1 if tabela_valid.loc[n, "acurácia valid. sem atalho (%)"] > acc4 + 1.0]
if len(ajudaram) >= 2:
    rc = dict(r4, arquivo="cnn_cnn5combinao")
    for n in ajudaram:
        rv = variacoes_r1[n]
        rc.update(canais=max(rc["canais"], rv["canais"]), dropout=max(rc["dropout"], rv["dropout"]), l2=max(rc["l2"], rv["l2"]),
                  aumento=rc["aumento"] or rv["aumento"], reduzir_lr=rc["reduzir_lr"] or rv["reduzir_lr"])
    nome_c = "R1 · combinação: " + " + ".join(n.split("· ")[1] for n in ajudaram)
    modelo, s, h, _ = rodar_receita(nome_c, rc, registrar_resultado=True, descricao="combinação das que ajudaram")
    modelos[nome_c], stats_de[nome_c], historicos[nome_c], receitas[nome_c] = modelo, s, h, rc

tabela_valid = pd.DataFrame(resultados).set_index("modelo")
display(tabela_valid)

# Regra da rodada 1: maior acurácia na validação sem atalho; a até 1 ponto, fica a mais simples (a primeira da lista)
candidatos = list(modelos)
acc = tabela_valid.loc[candidatos, "acurácia valid. sem atalho (%)"]
venc_r1 = [m for m in candidatos if acc[m] >= acc.max() - 1.0][0]
print("Variações que passaram a CNN 4 por mais de 1 ponto:", ajudaram or "nenhuma")
print("VENCEDORA DA RODADA 1:", venc_r1)

**Leitura:** a tabela inclui as CNNs 1 a 4 e os baselines, para a Seção 3. Compare a acurácia de treino com a de validação: a distância é o sobreajuste, que as próximas rodadas tentam reduzir. Dropout 0,5 e L2 não passaram a CNN 4: como alerta a Aula 5 (slides 22 e 31), mais regularização não é sempre melhor, e em excesso leva ao subajuste.

### 11.2 A vencedora se repete com outra semente?

A rodada 1 usou uma semente só. Repetimos a vencedora com mais duas: se o número balançar tanto quanto as diferenças entre as mudanças, a rodada 1 teve sorte, e as próximas rodadas precisam de 3 sementes.

In [ ]:
medir_sementes(venc_r1, receitas[venc_r1])
accs = [i["acurácia valid. sem atalho (%)"] for i in infos[venc_r1]]
print(f"{venc_r1} — sementes 42, 7 e 123: {accs} | média {np.mean(accs):.1f}% | de {min(accs)}% a {max(accs)}%")
ref1 = promover(venc_r1, 5, venc_r1.split("· ")[1]) if venc_r1.startswith("R1") else venc_r1
print("Referência da rodada 2:", ref1)

### 11.3 Rodada 2: três mudanças em cima da CNN 5

In [ ]:
r5 = receitas[ref1]
variacoes_r2 = {
    "R2 · deslocamento na frequência": {**r5, "desloc_freq": 0.05, "arquivo": "cnn_cnn4f4adeslocamentonafrequncia"},
    "R2 · taxa 0,0003":                {**r5, "taxa": 3e-4, "arquivo": "cnn_cnn4h4acomtaxa00003"},
    "R2 · lote 16":                    {**r5, "lote": 16, "arquivo": "cnn_cnn4i4acomlote16"},
}
tabela_r2 = rodada(ref1, variacoes_r2)
display(tabela_r2)
venc = vencedora_por_acuracia(tabela_r2, ref1, variacoes_r2)
ref2 = promover(venc, 6, venc.split("· ")[1]) if venc else ref1
print("VENCEDORA DA RODADA 2:", venc or "nenhuma (a referência continua)", "→", ref2)

**Leitura:** olhe a acurácia de **treino**. Com a taxa menor, a rede chega mais devagar ao melhor ponto e fica mais estável entre sementes. Com lote 16, a validação piora: as atualizações ficam ruidosas demais.

### 11.4 Rodada 3: três mudanças em cima da CNN 6

In [ ]:
r6 = receitas[ref2]
variacoes_r3 = {
    "R3 · aumento no tempo":             {**r6, "aumento": True, "arquivo": "cnn_cnn4m4haumentonotempo"},
    "R3 · taxa 0,001 que cai":           {**r6, "taxa": 0.001, "reduzir_lr": "acuracia", "arquivo": "cnn_cnn4o4hcomtaxaquecomeaem0001ecai"},
    "R3 · máscaras de tempo e frequência": {**r6, "mascaras": True, "arquivo": "cnn_cnn4p4hmscarasdetempoefrequncia"},
}
tabela_r3 = rodada(ref2, variacoes_r3)
display(tabela_r3)
venc = vencedora_por_acuracia(tabela_r3, ref2, variacoes_r3)
ref3 = promover(venc, 7, venc.split("· ")[1]) if venc else ref2
print("VENCEDORA DA RODADA 3:", venc or "nenhuma (a referência continua)", "→", ref3)

**Leitura:** nesta rodada, quase tudo fica a menos de 1 ponto da referência: a rede **chegou ao teto** de acurácia com estes dados. Os erros que sobram se concentram em classes de ruído de máquina com poucas gravações na base (Etapa 2.5), e mexer na rede não resolve isso.

A "taxa 0,001 que cai" é a taxa adaptativa da rodada 1, mas olhando a **acurácia** de validação (a mesma medida da parada antecipada), e não a perda, que sobe cedo (Etapa 7.1).

### 11.5 Rodada 4: o tiro com outro som junto · Seções 4 e 6

A coluna **"tiro com outro som (%)"** das tabelas mede quantas misturas do fold 9 (Etapa 4.2) a rede chama de tiro. Até a CNN 7, ela acha poucos: aprendeu o tiro **sozinho**. A rodada 4 põe as misturas dos folds de treino **no treino**:

| Mudança | O que entra no treino |
|---|---|
| tiros sobre outros sons (0, 6 e 12 dB) | as 3 misturas de cada tiro |
| tiros sobre outros sons (6 e 12 dB) | só as 2 em que o tiro está acima do outro som: um tiro no mesmo volume de um som contínuo é pouco realista e pode ensinar a rede a chamar de tiro qualquer batida |

**Mudança de critério, declarada aqui, antes do teste.** O objetivo do projeto é o alerta de tiro, e a acurácia das 10 classes não mede o caso de uso. Nesta rodada, a regra é:
1. a acurácia média não pode cair **mais de 1 ponto** em relação à referência (empate, pela regra das rodadas anteriores);
2. a precisão média do tiro isolado precisa ficar em **pelo menos 70%** (no máximo 3 alarmes falsos a cada 10);
3. entre as que cumprem 1 e 2, fica a de **maior recall do tiro com outro som**, desde que passe o da referência por **mais de 10 pontos**.

> ⚠️ **Transparência:** esta regra foi escrita **depois** de vermos a rodada 4 na validação, num protótipo. Nenhuma rede da rodada 4 foi avaliada no teste antes desta escolha.

In [ ]:
r7 = receitas[ref3]
variacoes_r4 = {
    "R4 · tiros sobre outros sons (0, 6 e 12 dB)": {**r7, "misturas": 0, "arquivo": "cnn_cnn4r4ptirossobreoutrossons"},
    "R4 · tiros sobre outros sons (6 e 12 dB)":    {**r7, "misturas": 6, "arquivo": "cnn_cnn4t4ptiros6e12dbacimadeoutrossons"},
}
tabela_r4 = rodada(ref3, variacoes_r4)
display(tabela_r4)

fundo_ref = tabela_r4.loc[ref3, "tiro com outro som (%)"]
cumprem = [n for n in variacoes_r4
           if tabela_r4.loc[n, "diferença p/ referência (pontos)"] >= -1.0      # 1. acurácia: empate ou melhor
           and tabela_r4.loc[n, "precisão tiro (%)"] >= 70                       # 2. precisão do tiro isolado
           and tabela_r4.loc[n, "tiro com outro som (%)"] > fundo_ref + 10]      # 3. ganho no caso de uso
for n in variacoes_r4:
    print(f"{n}: acurácia {tabela_r4.loc[n, 'diferença p/ referência (pontos)']:+.1f} pontos | "
          f"precisão tiro {tabela_r4.loc[n, 'precisão tiro (%)']:.1f}% | tiro com outro som "
          f"{tabela_r4.loc[n, 'tiro com outro som (%)']:.1f}% (referência {fundo_ref:.1f}%) → "
          f"{'cumpre' if n in cumprem else 'não cumpre'}")
venc = max(cumprem, key=lambda n: tabela_r4.loc[n, "tiro com outro som (%)"]) if cumprem else None
ref4 = promover(venc, 8, "tiros sobre outros sons") if venc else ref3
print("VENCEDORA DA RODADA 4:", venc or "nenhuma (a referência continua)", "→", ref4)

**Leitura:**
- **Tiro com outro som:** é aqui que a rodada 4 muda o jogo. Compare a coluna da referência com a das mudanças.
- **Precisão do tiro isolado:** com as misturas, a rede fica mais sensível a qualquer **estrondo**, e os alarmes falsos sobem; com o tiro no mesmo volume do outro som, esse efeito é maior.
- **Acurácia das 10 classes:** fica no mesmo patamar, porque as misturas só acrescentam exemplos de tiro.

### 11.6 Rodada 5: o que a CNN 8 ainda erra, e três aumentos de dados · Seções 4, 5 e 6

Antes de mudar a rede, medimos **na validação** três coisas que as tabelas anteriores não mostravam:

| Medida nova (fold 9) | Pergunta |
|---|---|
| **misturas sem tiro → tiro** | no treino da CNN 8, só o tiro aparece somado a outro som. A rede aprendeu "dois sons ao mesmo tempo = tiro"? Medimos com misturas de dois sons que **não** são tiro |
| **acurácia com microfone simulado** | e se o áudio vier de um microfone pior? Passamos a validação por dois filtros: telefone (300 a 3.400 Hz) e microfone barato (8.000 Hz) |
| **de onde vêm os alarmes falsos** | quais classes a rede chama de tiro sem ser |

Para isso (e para as mudanças da rodada), preparamos alguns dados novos a partir dos áudios. Tudo usa a mesma versão **sem atalho** de cada trecho (Etapa 5) e nada cruza entre folds:

| Dado | Como é feito | Para quê |
|---|---|---|
| misturas de dois sons sem tiro | para cada fold, tantas quanto as misturas de tiro com 6 e 12 dB (2 por tiro); o som principal fica 6 ou 12 dB acima do outro, que é de **outra classe e do mesmo fold**; o rótulo é o do principal | medir (fold 9) e treinar (folds 1 a 8) |
| trechos com bordas suavizadas | em 50% dos trechos curtos do treino, o começo e o fim sobem e descem aos poucos (10 a 100 ms, sorteados) | treinar |
| trechos com microfone simulado | em 30% dos trechos, um filtro passa-faixa sorteado (corte baixo de 100 a 500 Hz, alto de 3.000 a 8.000 Hz) | treinar |
| validação e teste com microfone simulado | telefone e 8.000 Hz, em todos os trechos dos folds 9 e 10 | medir |

**Demora:** cerca de 15 a 20 minutos na primeira vez (lê todos os áudios de novo); depois vem do cache.

In [ ]:
from scipy.signal import butter, sosfilt
ARQ_R5 = os.path.join(PASTA_CACHE, "us8k_r5.npz")

def suavizar(y, ms_inicio, ms_fim):
    # o começo sobe de 0 a 1 e o fim desce de 1 a 0, em ms_inicio e ms_fim milissegundos
    y = y.copy(); a, b = int(SR * ms_inicio / 1000), int(SR * ms_fim / 1000)
    a, b = min(a, len(y) // 2), min(b, len(y) // 2)
    if a > 0: y[:a] *= np.linspace(0, 1, a, dtype="float32")
    if b > 0: y[-b:] *= np.linspace(1, 0, b, dtype="float32")
    return y

def passa_faixa(y, baixo, alto):
    # filtro que só deixa passar as frequências entre baixo e alto (Hz), como um microfone pior
    return sosfilt(butter(4, [baixo, alto], btype="bandpass", fs=SR, output="sos"), y).astype("float32")

def telefone(y):
    return passa_faixa(y, 300, 3400)

def microfone_barato(y):
    # reduz a taxa para 8.000 Hz e volta: some tudo acima de 4.000 Hz
    y8 = librosa.resample(y, orig_sr=SR, target_sr=8000)
    return librosa.util.fix_length(librosa.resample(y8, orig_sr=8000, target_sr=SR), size=N_AMOSTRAS)

def sem_atalho_do_trecho(y, i, potencia=None):
    # a MESMA versão sem atalho da Etapa 5 (sorteio próprio do trecho i); o ruído usa a potência do som original
    rng = np.random.default_rng(i)
    saida = np.zeros(N_AMOSTRAS, dtype="float32"); inicio = rng.integers(0, N_AMOSTRAS - len(y) + 1)
    saida[inicio:inicio + len(y)] = y
    p = np.mean(y ** 2) if potencia is None else potencia
    return saida + rng.normal(0, np.sqrt(p / 10 ** (SNR_FUNDO / 10)), N_AMOSTRAS).astype("float32")

if os.path.exists(ARQ_R5):
    print("Dados da rodada 5 encontrados no cache")
else:
    R5 = {"XF": XB.copy(), "XC": XB.copy(), "FSUAV": np.zeros(len(meta), bool), "CMIC": np.zeros(len(meta), bool)}
    VT = np.where(meta["fold"].values >= 9)[0]; pos = {k: n for n, k in enumerate(VT)}
    R5["VT"] = VT; R5["XTEL"] = np.zeros((len(VT), N_MELS, N_QUADROS), "float16"); R5["X8K"] = R5["XTEL"].copy()
    for i, linha in enumerate(tqdm(meta.itertuples(), total=len(meta), desc="Rodada 5: trechos")):
        y = carregar_bruto(caminho(linha))
        r = np.random.default_rng(100000 + i)                     # bordas suavizadas: 50% dos curtos do treino
        if linha.fold <= 8 and len(y) < N_AMOSTRAS and r.random() < 0.5:
            R5["XF"][i] = espectrograma(sem_atalho_do_trecho(suavizar(y, r.uniform(10, 100), r.uniform(10, 100)), i, np.mean(y ** 2)))
            R5["FSUAV"][i] = True
        r = np.random.default_rng(200000 + i)                     # microfone simulado: 30% dos trechos
        if r.random() < 0.3:
            R5["XC"][i] = espectrograma(passa_faixa(sem_atalho_do_trecho(y, i), r.uniform(100, 500), r.uniform(3000, 8000)))
            R5["CMIC"][i] = True
        if linha.fold >= 9:                                       # validação e teste: os dois microfones fixos
            b = sem_atalho_do_trecho(y, i)
            R5["XTEL"][pos[i]] = espectrograma(telefone(b)); R5["X8K"][pos[i]] = espectrograma(microfone_barato(b))
    rng = np.random.default_rng(555); XN, ON, SEM_N = [], [], []  # ON: principal, outro, snr, classe principal, classe do outro, fold
    for fold in range(1, 11):
        n_tiros = int(((meta["fold"] == fold) & (meta["classID"] == TIRO)).sum())
        candidatos = np.where((meta["fold"].values == fold) & (meta["classID"].values != TIRO))[0]
        for k in tqdm(range(2 * n_tiros), desc=f"Misturas sem tiro, fold {fold}"):
            i = int(rng.choice(candidatos))
            j = int(rng.choice(candidatos[meta["classID"].values[candidatos] != meta["classID"].values[i]]))   # outra classe
            snr = 6 if k % 2 == 0 else 12; semente = int(rng.integers(1 << 31))
            onda = misturar(carregar_bruto(caminho(meta.iloc[i])), carregar_bruto(caminho(meta.iloc[j])), np.random.default_rng(semente), snr)
            XN.append(espectrograma(onda)); ON.append([i, j, snr, meta["classID"].values[i], meta["classID"].values[j], fold]); SEM_N.append(semente)
    R5.update(XN=np.array(XN, dtype="float16"), ON=np.array(ON), SEM_N=np.array(SEM_N))
    for k in ("XF", "XC"):
        R5[k] = R5[k].astype("float16")
    np.savez(ARQ_R5, **R5)
    print("Dados da rodada 5 calculados e salvos")
R5 = np.load(ARQ_R5)          # lido do disco quando cada parte é usada (economiza memória)
print(f"Trechos de treino com bordas suavizadas: {int(R5['FSUAV'].sum())} | com microfone simulado (folds 1 a 8): "
      f"{int(R5['CMIC'][treino].sum())} | misturas sem tiro: {len(R5['XN'])} (treino: {int((R5['ON'][:, 5] <= 8).sum())})")

In [ ]:
# Diagnóstico na validação: as medidas novas para a CNN 7 e a CNN 8 (3 sementes; os pesos vêm do cache)
for ref in (ref3, ref4):
    medir_sementes(ref, receitas[ref])
colunas_diag = ["média (%)", "precisão tiro (%)", "tiro com outro som (%)", "misturas sem tiro → tiro (%)",
                "acurácia c/ microfone simulado (%)", "precisão tiro c/ microfone simulado (%)"]
diagnostico = pd.DataFrame([linha_rodada(ref3), linha_rodada(ref4)]).set_index("rede")[colunas_diag]
display(diagnostico)
for ref in (ref3, ref4):
    p = prever(modelos[ref], montar(XB[valid], stats_de[ref])).argmax(1)
    falsos = pd.Series(y_valid[(p == TIRO) & (y_valid != TIRO)]).map(dict(enumerate(ROTULOS))).value_counts()
    print(f"{ref} (semente 42) — alarmes falsos de tiro na validação vêm de:", {k: int(v) for k, v in falsos.items()})

**Leitura do diagnóstico:**
- **Misturas sem tiro → tiro:** se for baixo, a rede **não** aprendeu "dois sons = tiro"; o atalho que temíamos é pequeno.
- **Microfone simulado:** compare a CNN 7 com a CNN 8. Treinar com tiros misturados deixou a rede mais **frágil** a um microfone pior: com o som abafado, ela passa a chamar quase tudo de tiro (veja a precisão do tiro com microfone simulado). Era um custo escondido da rodada 4.
- **Alarmes falsos:** na CNN 8, eles vêm sobretudo de **ar-condicionado**, um ruído contínuo parecido com o "fundo" das misturas.

**As três mudanças da rodada 5** (cada uma muda só uma coisa em cima da CNN 8; todas são aumento de dados, Aula 4, e seguem a regra da aumentação da Aula 5: a classe continua a mesma):

| Mudança | O que entra no treino | O que ataca |
|---|---|---|
| **R5 · misturas de sons sem tiro** | as misturas sem tiro dos folds 1 a 8, com o rótulo do som principal | o possível atalho "dois sons = tiro" |
| **R5 · bordas suavizadas** | os trechos curtos com bordas suavizadas no lugar dos originais | o começo seco do trecho como pista de tiro (Etapas 13.8 e 13.9) |
| **R5 · microfones simulados** | os trechos com microfone simulado no lugar dos originais | a fragilidade a microfones piores |

**Regra da rodada 5, escrita antes de treinar.** Referência: CNN 8 (média das 3 sementes). Uma mudança passa se, na média:
1. acurácia ≥ referência − 1 ponto;
2. recall do tiro ≥ referência − 3,3 pontos (no máximo ~1 tiro a menos, dos 31);
3. tiro com outro som ≥ referência − 5 pontos;
4. e cumprir **o seu objetivo**: misturas sem tiro e bordas → precisão do tiro ≥ referência + 5 pontos **ou** acurácia > referência + 1 ponto; microfones → acurácia com microfone simulado ≥ referência + 10 pontos.

Se mais de uma passar, treinamos a **combinação** (3 sementes): ela fica se cumprir 1 a 3 e os objetivos de cada parte; senão, fica a mudança de maior acurácia. Se nenhuma passar, a CNN 8 continua.

> ⚠️ **Transparência:** as ideias das bordas e dos microfones surgiram da análise de erros (Etapas 13.7 a 13.9) de uma **versão anterior** deste notebook, na qual o teste já tinha sido olhado. Ter uma ideia olhando os erros é normal; o que não pode é **decidir** olhando o teste. Por isso a decisão é feita só na validação, com a regra acima escrita antes de treinar. A validação cruzada (Etapa 13.6) é a medida que não depende do fold 10.

In [ ]:
r8 = receitas[ref4]
variacoes_r5 = {
    "R5 · misturas de sons sem tiro": {**r8, "misturas_sem_tiro": True, "arquivo": "cnn_r5a_misturas_nao_tiro"},
    "R5 · bordas suavizadas":         {**r8, "bordas": True, "arquivo": "cnn_r5b_bordas_suaves"},
    "R5 · microfones simulados":      {**r8, "microfones": True, "arquivo": "cnn_r5c_microfones"},
}
objetivo_r5 = {"R5 · misturas de sons sem tiro": "precisão ou acurácia", "R5 · bordas suavizadas": "precisão ou acurácia",
               "R5 · microfones simulados": "microfone"}
tabela_r5 = rodada(ref4, variacoes_r5)
R = tabela_r5.loc[ref4]

def cumpre_minimos(linha):
    return (linha["média (%)"] >= R["média (%)"] - 1.0                          # 1. acurácia
            and linha["recall tiro (%)"] >= R["recall tiro (%)"] - 3.3          # 2. recall do tiro
            and linha["tiro com outro som (%)"] >= R["tiro com outro som (%)"] - 5.0)   # 3. tiro com outro som

def cumpre_objetivo(linha, objetivo):
    if objetivo == "microfone":
        return linha["acurácia c/ microfone simulado (%)"] >= R["acurácia c/ microfone simulado (%)"] + 10
    return linha["precisão tiro (%)"] >= R["precisão tiro (%)"] + 5 or linha["média (%)"] > R["média (%)"] + 1

passaram = []
for n in variacoes_r5:
    ok1, ok2 = cumpre_minimos(tabela_r5.loc[n]), cumpre_objetivo(tabela_r5.loc[n], objetivo_r5[n])
    print(f"{n}: mínimos {'ok' if ok1 else 'NÃO'} | objetivo ({objetivo_r5[n]}) {'ok' if ok2 else 'NÃO'}")
    if ok1 and ok2:
        passaram.append(n)

venc_r5 = None
if len(passaram) >= 2:                      # a combinação das que passaram
    letra = {"misturas_sem_tiro": "a", "bordas": "b", "microfones": "c"}
    rc = dict(r8)
    for n in passaram:
        rc.update({k: True for k in letra if variacoes_r5[n].get(k)})
    rc["arquivo"] = "cnn_r5_" + "_".join(letra[k] for k in letra if rc.get(k))
    nome_c = "R5 · combinação: " + " + ".join(n.split("· ")[1] for n in passaram)
    medir_sementes(nome_c, rc); receitas[nome_c] = rc
    tabela_r5 = pd.concat([tabela_r5, pd.DataFrame([linha_rodada(nome_c, R["média (%)"])]).set_index("rede")])
    linha_c = tabela_r5.loc[nome_c]
    ok = cumpre_minimos(linha_c) and all(cumpre_objetivo(linha_c, objetivo_r5[n]) for n in passaram)
    print(f"{nome_c}: {'cumpre os mínimos e os objetivos de cada parte' if ok else 'não cumpre'}")
    venc_r5 = nome_c if ok else tabela_r5.loc[passaram, "média (%)"].idxmax()
elif passaram:
    venc_r5 = passaram[0]
display(tabela_r5)

if venc_r5:
    ideia = "misturas sem tiro e microfones simulados" if venc_r5.startswith("R5 · combinação") else venc_r5.split("· ")[1]
    escolha = promover(venc_r5, 9, ideia)
else:
    escolha = ref4
modelo_final, stats_final = modelos[escolha], stats_de[escolha]
print("VENCEDORA DA RODADA 5:", venc_r5 or "nenhuma (a CNN 8 continua)", "→", escolha)

**Leitura:**
- **Misturas de sons sem tiro:** a acurácia sobe; a rede aprende que outras classes também aparecem com outro som junto.
- **Bordas suavizadas:** não reduz os alarmes falsos de forma consistente entre sementes; o ar-condicionado continua sendo chamado de tiro.
- **Microfones simulados:** o ganho com microfone pior é grande, sem perder acurácia na condição normal. O recall do tiro fica perto do limite da regra: olhem as sementes.
- **Combinação:** junta os dois ganhos. A precisão do tiro cai um pouco em relação à CNN 8 (a regra da rodada 5 não tinha piso de precisão; a da rodada 4 tinha 70%).
- **O que nenhuma resolveu:** os alarmes falsos de ar-condicionado.

### 11.7 Rodada 6: transferência de aprendizado com uma rede que já ouviu tiros · Seções 3 e 4

**Transferência de aprendizado** (Aula 4): em vez de aprender tudo do zero com 7 mil trechos, usamos uma rede já treinada em uma base enorme e treinamos só a parte final.

**Por que não uma rede de imagens?** No protótipo, a MobileNetV2 treinada no ImageNet (fotos) perdeu para a nossa CNN (Etapa 11, quadro inicial). Ela sabe reconhecer bordas e texturas de foto; no espectrograma, a altura é a frequência e tem significado, e ela nunca ouviu um som.

**A YAMNet** (Google) foi treinada no **AudioSet**: cerca de 2 milhões de trechos do YouTube, em **521 tipos de som**, inclusive *tiro*, *metralhadora*, *fogos de artifício* e *bombinha*. O AudioSet vem do YouTube e o UrbanSound8K do Freesound: não há os mesmos áudios nas duas bases.

| | |
|---|---|
| **Entrada da YAMNet** | o áudio em 16.000 Hz (a versão sem atalho de cada trecho, a mesma da CNN) |
| **O que usamos dela** | os 1.024 números que ela calcula para cada janela de ~1 s; resumimos as janelas do trecho pela **média** e pelo **máximo** (o máximo guarda eventos curtos, como o tiro): 2.048 números |
| **YAMNet** | **congelada**: nenhum peso dela é treinado |
| **Cabeças treinadas em Keras** | R6 · `Dense(10, softmax)` direto; R6 · `Dense(256, relu)` + `Dropout(0.3)` + `Dense(10, softmax)` |
| **Receita** | a mesma da CNN 8: treino com os tiros 6 e 12 dB sobre outros sons; padronização com estatísticas só do treino; `EarlyStopping` na acurácia de validação |

**Regra da rodada 6, escrita antes:** entre as duas cabeças, fica a de maior acurácia média (3 sementes). Ela substitui a CNN 9 se: acurácia > CNN 9 + 1 ponto (trocar de arquitetura precisa ganhar, como nas rodadas 1 a 3), recall do tiro ≥ CNN 9 − 3,3 pontos, tiro com outro som ≥ CNN 9 − 5 pontos e precisão do tiro ≥ 70%.

**Demora:** no Colab, a YAMNet é baixada do TensorFlow Hub. Calcular os vetores de todos os trechos e misturas leva cerca de 25 minutos na primeira vez; depois vem do cache. As cabeças treinam em segundos.

In [ ]:
import sys
ARQ_YAMNET = os.path.join(PASTA_CACHE, "us8k_yamnet.npz")

def carregar_yamnet():
    try:
        import tensorflow_hub as hub
        return hub.load("https://tfhub.dev/google/yamnet/1")
    except Exception as erro:                 # sem internet para o TF Hub: arquivos baixados à mão, na pasta do cache
        print("TF Hub indisponível (", erro, "); usando os arquivos em", os.path.join(PASTA_CACHE, "yamnet"))
        sys.path.insert(0, os.path.join(PASTA_CACHE, "yamnet"))
        import params as yamnet_params, yamnet as yamnet_codigo
        m = yamnet_codigo.yamnet_frames_model(yamnet_params.Params())
        m.load_weights(os.path.join(PASTA_CACHE, "yamnet", "yamnet.h5"))
        return m

def vetor_yamnet(onda):
    # onda em 22.050 Hz → 16.000 Hz → YAMNet → média e máximo das janelas (2.048 números)
    y16 = librosa.resample(onda, orig_sr=SR, target_sr=16000).astype("float32")
    _, janelas, _ = yamnet(y16)
    janelas = np.asarray(janelas)
    return np.concatenate([janelas.mean(0), janelas.max(0)]).astype("float32")

if os.path.exists(ARQ_YAMNET):
    YA = dict(np.load(ARQ_YAMNET))
    print("Vetores da YAMNet carregados do cache:", YA["E"].shape)
else:
    yamnet = carregar_yamnet()
    rng = np.random.default_rng(2024); EM = []        # as MESMAS misturas de tiro da Etapa 4.2 (mesmos sorteios)
    for fold in range(1, 11):
        tiros = np.where((meta["fold"].values == fold) & (meta["classID"].values == TIRO))[0]
        fundos = np.where((meta["fold"].values == fold) & (meta["classID"].values != TIRO))[0]
        for i in tqdm(tiros, desc=f"YAMNet: misturas de tiro, fold {fold}"):
            y_tiro = carregar_bruto(caminho(meta.iloc[i]))
            for snr in SNRS_MISTURA:
                j = int(rng.choice(fundos))
                EM.append(vetor_yamnet(misturar(y_tiro, carregar_bruto(caminho(meta.iloc[j])), rng, snr)))
    EN = [vetor_yamnet(misturar(carregar_bruto(caminho(meta.iloc[i])), carregar_bruto(caminho(meta.iloc[j])),
                                np.random.default_rng(int(sem)), snr))                       # as mesmas misturas sem tiro da 11.6
          for (i, j, snr, _, _, _), sem in tqdm(list(zip(R5["ON"], R5["SEM_N"])), desc="YAMNet: misturas sem tiro")]
    E = np.zeros((len(meta), 2048), "float32")
    ETEL, E8K = np.zeros((len(R5["VT"]), 2048), "float32"), np.zeros((len(R5["VT"]), 2048), "float32")
    pos = {k: n for n, k in enumerate(R5["VT"])}
    for i, linha in enumerate(tqdm(meta.itertuples(), total=len(meta), desc="YAMNet: trechos")):
        b = sem_atalho_do_trecho(carregar_bruto(caminho(linha)), i)
        E[i] = vetor_yamnet(b)
        if i in pos:
            ETEL[pos[i]], E8K[pos[i]] = vetor_yamnet(telefone(b)), vetor_yamnet(microfone_barato(b))
    YA = dict(E=E, EM=np.array(EM), EN=np.array(EN), VT=R5["VT"], ETEL=ETEL, E8K=E8K)
    np.savez(ARQ_YAMNET, **YA)
    print("Vetores da YAMNet calculados e salvos:", E.shape)

In [ ]:
# Cabeças em Keras sobre a YAMNet congelada: a mesma receita da CNN 8 (tiros 6 e 12 dB no treino)
usar_m = (OM[:, 4] <= 8) & (OM[:, 2] >= 6)
Y_tr = np.concatenate([YA["E"][treino], YA["EM"][usar_m]]); yy_tr = np.concatenate([y_treino, np.full(int(usar_m.sum()), TIRO)])
media_y, desvio_y = Y_tr.mean(0), Y_tr.std(0) + 1e-6          # padronização com estatísticas só do treino
pad_y = lambda X: (X - media_y) / desvio_y
no9 = meta["fold"].values[YA["VT"]] == 9

def cabeca_yamnet(variante, semente):
    keras.utils.set_random_seed(semente)
    camadas = [keras.Input(shape=(2048,))]
    if variante == "6b":
        camadas += [layers.Dense(256, activation="relu"), layers.Dropout(0.3)]
    camadas.append(layers.Dense(10, activation="softmax"))
    m = keras.Sequential(camadas, name=f"yamnet_{variante}_s{semente}")
    m.compile(optimizer=keras.optimizers.Adam(0.001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    arq = os.path.join(PASTA_MODELOS, f"{m.name}.weights.h5")
    if os.path.exists(arq) and not RETREINAR:
        m.load_weights(arq)
    else:
        m.fit(pad_y(Y_tr), yy_tr, validation_data=(pad_y(YA["E"][valid]), y_valid), epochs=100, batch_size=32, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=8, restore_best_weights=True)])
        m.save_weights(arq)
    return m

nomes_r6 = {"6a": "R6 · YAMNet + Dense(10)", "6b": "R6 · YAMNet + Dense(256) + Dense(10)"}
for variante, nome in nomes_r6.items():
    lista = []
    for semente in SEMENTES:
        m = cabeca_yamnet(variante, semente)
        P = lambda X: prever(m, pad_y(X)).argmax(1)
        met = metricas(y_valid, P(YA["E"][valid]))
        p_mic = [P(YA[k][no9]) for k in ("ETEL", "E8K")]
        lista.append({"acurácia valid. sem atalho (%)": met["acurácia (%)"], "F1-macro (%)": met["F1-macro (%)"],
                      "recall tiro (%)": met["recall tiro (%)"], "precisão tiro (%)": met["precisão tiro (%)"],
                      "tiro com outro som (%)": round(float(100 * (P(YA["EM"][OM[:, 4] == 9]) == TIRO).mean()), 1),
                      "acurácia treino (%)": round(float(100 * (P(Y_tr) == yy_tr).mean()), 1),
                      "misturas sem tiro → tiro (%)": round(float(100 * (P(YA["EN"][R5["ON"][:, 5] == 9]) == TIRO).mean()), 1),
                      "acurácia c/ microfone simulado (%)": round(float(np.mean([100 * (p == y_valid).mean() for p in p_mic])), 1),
                      "precisão tiro c/ microfone simulado (%)": round(float(np.mean([100 * (y_valid[p == TIRO] == TIRO).mean() if (p == TIRO).any() else 0 for p in p_mic])), 1),
                      "melhor época": 0, "parâmetros": m.count_params()})
    infos[nome] = lista

media_ref = float(np.mean([i["acurácia valid. sem atalho (%)"] for i in infos[escolha]]))
tabela_r6 = pd.DataFrame([linha_rodada(escolha)] + [linha_rodada(n, media_ref) for n in nomes_r6.values()]).set_index("rede")
tabela_r6 = tabela_r6.drop(columns=["melhores épocas"]).rename(columns={"parâmetros": "parâmetros treinados"})
display(tabela_r6)

melhor_r6 = tabela_r6.loc[list(nomes_r6.values()), "média (%)"].idxmax()
L, C = tabela_r6.loc[melhor_r6], tabela_r6.loc[escolha]
regras_r6 = {"acurácia > CNN 9 + 1 ponto": L["média (%)"] > C["média (%)"] + 1,
             "recall do tiro ≥ CNN 9 − 3,3": L["recall tiro (%)"] >= C["recall tiro (%)"] - 3.3,
             "tiro com outro som ≥ CNN 9 − 5": L["tiro com outro som (%)"] >= C["tiro com outro som (%)"] - 5,
             "precisão do tiro ≥ 70%": L["precisão tiro (%)"] >= 70}
print("Melhor cabeça:", melhor_r6)
for regra, ok in regras_r6.items():
    print(f"  {regra}: {'ok' if ok else 'NÃO'}")
print("RODADA 6:", f"{melhor_r6} substitui a {escolha}" if all(regras_r6.values()) else f"a YAMNet não é adotada; fica a {escolha}")

**Leitura:**
- **Com uma camada só** (20 mil parâmetros treinados), a YAMNet chega perto da nossa CNN: é o argumento clássico da transferência. O que ela aprendeu no AudioSet já separa bem os sons urbanos.
- **Microfone simulado:** ela também é robusta a um microfone pior, porque ouviu áudio de todo tipo de microfone no YouTube.
- **Precisão do tiro:** é onde ela perde. Ela chama de tiro mais coisas (furadeira, crianças, música de rua). Faz sentido: no AudioSet, *tiro*, *explosão* e *fogos de artifício* ficam muito próximos, e a cabeça não consegue separá-los só com os nossos 7 mil trechos.
- **Com mais tempo:** ajustar as últimas camadas da YAMNet (ajuste fino), ou usar as próprias probabilidades dela de *fogos* e *bombinha* como alerta de "parece tiro, mas pode ser fogos", o maior risco de uso real (Etapa 15).

**Conclusão:** testamos a transferência de aprendizado com a YAMNet e ficamos com a nossa CNN, porque a YAMNet não trouxe melhora relevante: ficou abaixo na acurácia e teve **precisão do tiro pior** (cerca de 61%, contra 70% da CNN 9), com a cabeça maior treinando 4 vezes mais parâmetros que a nossa rede inteira.

### 11.8 Rodada 7: RawBoost no lugar dos microfones simulados · Seções 4 e 6

O **RawBoost** (Tak e outros, ICASSP 2022) é um aumento de dados criado para detectar voz falsificada. Ele distorce a **onda** (antes do espectrograma), com sorteios a cada trecho:

| Parte | O que faz | Imita |
|---|---|---|
| (1) convolutiva, linear e não linear | passa o som por filtros com 5 faixas sorteadas, também elevado a potências | microfone, canal de transmissão, distorção do aparelho |
| (2) impulsiva | até 10% das amostras ganham um "estalo" proporcional ao próprio som | cliques, saturação momentânea |
| (3) aditiva estacionária | ruído colorido somado, com relação sinal-ruído de 10 a 40 dB | ruído de fundo |

Usamos o **código dos autores** (licença MIT), com os parâmetros padrão deles. Primeiro, a pergunta direta: o RawBoost faz melhor o papel dos microfones simulados? Ele entra **no lugar** deles, **nos mesmos 30% de trechos** (só muda o tipo de distorção):
- **RawBoost completo:** (1) → (2) → (3);
- **RawBoost sem o impulsivo:** (1) → (3). Hipótese declarada antes: os estalos se parecem com tiro e podem aumentar os alarmes falsos.

**Regra (escrita antes):** referência = CNN 9. Entra se: acurácia ≥ ref. − 1; recall do tiro ≥ ref. − 3,3; tiro com outro som ≥ ref. − 5; acurácia com microfone simulado ≥ ref. − 5 (não perder a robustez); **e** ganhar em algo: acurácia > ref. + 1 **ou** precisão do tiro ≥ ref. + 5.

**Demora:** preparar as distorções leva cerca de 25 minutos na primeira vez (depois vem do cache).

In [ ]:
# RawBoost: baixamos o código dos autores (github.com/TakHemlata/SSL_Anti-spoofing, licença MIT)
ARQ_R7, ARQ_R8 = os.path.join(PASTA_CACHE, "us8k_r7.npz"), os.path.join(PASTA_CACHE, "us8k_r8.npz")
if not (os.path.exists(ARQ_R7) and os.path.exists(ARQ_R8)):
    import urllib.request, sys
    arq_rb = os.path.join(PASTA_CACHE, "RawBoost.py")
    urllib.request.urlretrieve("https://raw.githubusercontent.com/TakHemlata/SSL_Anti-spoofing/main/RawBoost.py", arq_rb)
    texto = open(arq_rb).read().replace("size=(1,))", "size=(1,))[0]")   # única mudança: numpy novo não aceita int() de uma lista
    open(arq_rb, "w").write(texto)
    sys.path.insert(0, PASTA_CACHE)
    from RawBoost import LnL_convolutive_noise, ISD_additive_noise, SSI_additive_noise
    F = (5, 20, 8000, 100, 1000, 10, 100, 0, 0)          # nBands, minF, maxF, minBW, maxBW, minCoeff, maxCoeff, minG, maxG

    def rawboost(x, impulsivo=True):
        # (1) → (2) → (3) em série ("algo 4" dos autores); sem o impulsivo, (1) → (3) ("algo 6")
        x = LnL_convolutive_noise(x, 5, *F, 5, 20, SR)
        if impulsivo:
            x = ISD_additive_noise(x, 10, 2)
        return SSI_additive_noise(x, 10, 40, *F, SR)

    def preparar(indices, sementes, arquivo):
        XR4 = np.zeros((len(indices), N_MELS, N_QUADROS), "float16"); XR6 = XR4.copy()
        for n, i in enumerate(tqdm(indices, desc=os.path.basename(arquivo))):
            b = sem_atalho_do_trecho(carregar_bruto(caminho(meta.iloc[i])), i).astype("float64")
            np.random.seed(sementes[0] + i); XR4[n] = espectrograma(rawboost(b, impulsivo=True).astype("float32"))
            np.random.seed(sementes[1] + i); XR6[n] = espectrograma(rawboost(b, impulsivo=False).astype("float32"))
        np.savez(arquivo, IDX=indices, XR4=XR4, XR6=XR6)

    CMIC = R5["CMIC"]
    preparar(np.where(CMIC)[0], (300000, 400000), ARQ_R7)              # rodada 7: os mesmos trechos do microfone
    outros = np.array([i for i in range(len(meta)) if not CMIC[i] and np.random.default_rng(500000 + i).random() < 3 / 7])
    preparar(outros, (600000, 700000), ARQ_R8)                          # rodada 8: ~30% dos trechos, sem microfone
R7, R8 = np.load(ARQ_R7), np.load(ARQ_R8)
print(f"RawBoost — rodada 7: {len(R7['IDX'])} trechos (os do microfone simulado) | rodada 8: {len(R8['IDX'])} outros trechos")

In [ ]:
r9 = receitas[escolha]
variacoes_r7 = {
    "R7 · RawBoost completo no lugar dos microfones": {**r9, "microfones": False, "rawboost": "completo", "arquivo": "cnn_r7a_rawboost_completo"},
    "R7 · RawBoost sem impulsivo no lugar dos microfones": {**r9, "microfones": False, "rawboost": "sem_impulsivo", "arquivo": "cnn_r7b_rawboost_sem_impulsivo"},
}
tabela_r7 = rodada(escolha, variacoes_r7)
display(tabela_r7)
R = tabela_r7.loc[escolha]
for n in variacoes_r7:
    L = tabela_r7.loc[n]
    minimos = (L["média (%)"] >= R["média (%)"] - 1 and L["recall tiro (%)"] >= R["recall tiro (%)"] - 3.3
               and L["tiro com outro som (%)"] >= R["tiro com outro som (%)"] - 5
               and L["acurácia c/ microfone simulado (%)"] >= R["acurácia c/ microfone simulado (%)"] - 5)
    ganho = L["média (%)"] > R["média (%)"] + 1 or L["precisão tiro (%)"] >= R["precisão tiro (%)"] + 5
    print(f"{n}: mínimos {'ok' if minimos else 'NÃO'} | ganho {'ok' if ganho else 'NÃO'}")

**Leitura:** no lugar dos microfones simulados, o RawBoost **melhora a precisão do tiro** (menos alarmes falsos), mas **perde a robustez ao microfone simulado**, e o completo perde também mais de 1 ponto de acurácia. Os microfones simulados foram feitos sob medida para o nosso risco (cortar graves e agudos); o RawBoost espalha o esforço em outras distorções. Os estalos (versão completa) pioraram o microfone ainda mais, como a hipótese previa. Ele não substitui os microfones, mas pode **somar-se** a eles: é a rodada 8.

### 11.9 Rodada 8: RawBoost junto com os microfones simulados · Seções 4 e 6

**Motivo (uso real):** os dois aumentos cobrem coisas diferentes. Os microfones simulados cobrem o microfone abafado; o RawBoost cobre canal de transmissão, ruído de fundo e estalos, distorções que o nosso teste não mede, e na rodada 7 trouxe precisão do tiro. A receita da CNN 9 fica igual e o RawBoost entra em **outros** trechos: entre os ~70% que não receberam microfone simulado, sorteamos 3/7, para que ~30% do treino tenha RawBoost. Ficam ~30% com microfone, ~30% com RawBoost e ~40% limpos.

**Regra (escrita antes), de não inferioridade:** como o RawBoost entra para proteger contra distorções que não medimos, basta que ele **não piore** o que já foi validado. Referência = CNN 9. Entra se: acurácia ≥ ref. − 1; recall do tiro ≥ ref. − 3,3; tiro com outro som ≥ ref. − 5; acurácia com microfone simulado ≥ ref. − 5. Se as duas versões passarem, fica a de maior acurácia; ela vira a **CNN 10**.

In [ ]:
variacoes_r8 = {
    "R8 · + RawBoost completo": {**r9, "rawboost": "completo", "rawboost_onde": "junto", "arquivo": "cnn_r8a_mic_rawboost_completo"},
    "R8 · + RawBoost sem impulsivo": {**r9, "rawboost": "sem_impulsivo", "rawboost_onde": "junto", "arquivo": "cnn_r8b_mic_rawboost_sem_impulsivo"},
}
tabela_r8 = rodada(escolha, variacoes_r8)
display(tabela_r8)
R = tabela_r8.loc[escolha]
passaram_r8 = []
for n in variacoes_r8:
    L = tabela_r8.loc[n]
    ok = (L["média (%)"] >= R["média (%)"] - 1 and L["recall tiro (%)"] >= R["recall tiro (%)"] - 3.3
          and L["tiro com outro som (%)"] >= R["tiro com outro som (%)"] - 5
          and L["acurácia c/ microfone simulado (%)"] >= R["acurácia c/ microfone simulado (%)"] - 5)
    print(f"{n}: {'não piora: entra' if ok else 'piora: não entra'}")
    if ok:
        passaram_r8.append(n)
venc_r8 = max(passaram_r8, key=lambda n: tabela_r8.loc[n, "média (%)"]) if passaram_r8 else None
if venc_r8:
    escolha = promover(venc_r8, 10, "RawBoost" + (" sem impulsivo" if "sem impulsivo" in venc_r8 else ""))
    modelo_final, stats_final = modelos[escolha], stats_de[escolha]
print("VENCEDORA DA RODADA 8:", venc_r8 or "nenhuma (a CNN 9 continua)", "→", escolha)

**Leitura:** mesmo junto com os microfones simulados, o RawBoost piorou o que já estava validado, e pela regra não entra: a CNN 9 continua sendo a final. Fica registrado como testado.

**Desempate em condições adversas novas.** O RawBoost entra para proteger contra distorções do mundo real, e essas distorções não estão nas medidas acima. Para não decidir no escuro, medimos os modelos em três condições que **nenhum deles viu no treino**, aplicadas à validação sem atalho (fold 9). Usar o próprio RawBoost para medir seria injusto: os modelos com RawBoost treinaram com ele.

| Condição | O que imita |
|---|---|
| MP3 a 24 kbit/s | transmissão ou gravação comprimida, de baixa qualidade |
| eco (resposta de sala sintética, 0,4 a 1,0 s) | som gravado longe, entre paredes e prédios |
| falhas de conexão (10% do áudio zerado em blocos de 20 a 60 ms) | perda de pacotes num sensor em rede |

**Regra (escrita antes):** a versão com RawBoost substitui a CNN 9 se, na média das três condições e das 3 sementes, acertar **mais de 2 pontos acima** dela (folga maior que a variação entre sementes) e não perder mais de 3,3 pontos de recall do tiro. A CNN 8, que treinou sem os microfones simulados, entra só como referência.

**Demora:** uns 15 minutos na primeira vez (o MP3 usa o `ffmpeg`, que já vem no Colab); depois vem do cache.

In [ ]:
import subprocess, tempfile, soundfile as sf
from scipy.signal import fftconvolve
ARQ_ADV = os.path.join(PASTA_CACHE, "us8k_adversas_valid.npz")

def mp3_baixa_qualidade(z):
    with tempfile.TemporaryDirectory() as pasta:
        a, b = os.path.join(pasta, "a.wav"), os.path.join(pasta, "b.mp3")
        sf.write(a, z / max(1e-9, np.abs(z).max()) * 0.9, SR)
        subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", a, "-ac", "1", "-ar", "16000", "-b:a", "24k", b], check=True)
        y, _ = librosa.load(b, sr=SR, mono=True)
    return librosa.util.fix_length(y, size=N_AMOSTRAS).astype("float32")

def eco(z, i):
    r = np.random.default_rng(800000 + i); rt60 = r.uniform(0.4, 1.0); n = int(SR * rt60)
    h = r.normal(0, 1, n) * np.exp(-6.9 * np.arange(n) / n); h[0] = 1.0 / 0.3      # som direto + cauda que decai
    y = fftconvolve(z, h * 0.3)[:N_AMOSTRAS]
    return (y / max(1e-9, np.abs(y).max()) * np.abs(z).max()).astype("float32")

def falhas_de_conexao(z, i):
    r = np.random.default_rng(900000 + i); y = z.copy(); alvo, zerado = int(0.10 * N_AMOSTRAS), 0
    while zerado < alvo:
        L = int(SR * r.uniform(0.02, 0.06)); ini = r.integers(0, N_AMOSTRAS - L); y[ini:ini + L] = 0; zerado += L
    return y

if not os.path.exists(ARQ_ADV):
    idx = np.where(valid)[0]; ADV = {k: np.zeros((len(idx), N_MELS, N_QUADROS), "float16") for k in ("MP3", "ECO", "FALHAS")}
    for n, i in enumerate(tqdm(idx, desc="Condições adversas (validação)")):
        b = sem_atalho_do_trecho(carregar_bruto(caminho(meta.iloc[i])), i)
        ADV["MP3"][n] = espectrograma(mp3_baixa_qualidade(b))
        ADV["ECO"][n] = espectrograma(eco(b, i))
        ADV["FALHAS"][n] = espectrograma(falhas_de_conexao(b, i))
    np.savez(ARQ_ADV, IDX=idx, **ADV)
ADV = np.load(ARQ_ADV)

def carregar_rede(r, semente):
    # a rede já treinada (rodadas acima) com aquela semente, direto do cache
    nome = r["arquivo"] + ("" if semente == 42 else f"_s{semente}")
    m = criar_cnn(nome, dropout=r["dropout"], canais=r["canais"], l2=r["l2"], aumento=r["aumento"],
                  desloc_freq=r.get("desloc_freq", 0.0), mascaras=r.get("mascaras", False), semente=semente)
    m.load_weights(os.path.join(PASTA_MODELOS, f"{nome}_{VERSAO_TREINO}.weights.h5"))
    return m

s3c = stats_por_canais[3]
condicoes_adv = {"MP3 24 kbit/s": montar(ADV["MP3"], s3c), "eco": montar(ADV["ECO"], s3c), "falhas de conexão": montar(ADV["FALHAS"], s3c)}
concorrentes = {"CNN 8 (referência, sem microfones simulados)": receitas[ref4], escolha: receitas[escolha],
                **{n.replace("R8 · ", "CNN 9 "): r for n, r in variacoes_r8.items()}}
linhas_adv = []
for nome, r in concorrentes.items():
    for semente in SEMENTES:
        m = carregar_rede(r, semente)
        for cond, X in condicoes_adv.items():
            p = prever(m, X).argmax(1); mt = metricas(y_valid, p)
            linhas_adv.append({"modelo": nome, "semente": semente, "condição": cond, "acurácia (%)": mt["acurácia (%)"],
                               "recall tiro (%)": mt["recall tiro (%)"], "precisão tiro (%)": mt["precisão tiro (%)"]})
adv = pd.DataFrame(linhas_adv)
tabela_adversas = adv.pivot_table(index="modelo", columns="condição", values="acurácia (%)", aggfunc="mean").round(1)
tabela_adversas["média das 3 (%)"] = adv.groupby("modelo")["acurácia (%)"].mean().round(1)
tabela_adversas["recall tiro, média (%)"] = adv.groupby("modelo")["recall tiro (%)"].mean().round(1)
tabela_adversas["precisão tiro, média (%)"] = adv.groupby("modelo")["precisão tiro (%)"].mean().round(1)
tabela_adversas = tabela_adversas.loc[list(concorrentes)]
display(tabela_adversas)
C9 = tabela_adversas.loc[escolha]
for n in list(concorrentes)[2:]:
    L = tabela_adversas.loc[n]
    ok = L["média das 3 (%)"] > C9["média das 3 (%)"] + 2 and L["recall tiro, média (%)"] >= C9["recall tiro, média (%)"] - 3.3
    print(f"{n}: {L['média das 3 (%)'] - C9['média das 3 (%)']:+.1f} pontos na média das condições adversas → "
          f"{'substitui a ' + escolha if ok else 'não substitui'}")

**Leitura:**
- **O RawBoost ajuda pouco nestas condições:** as versões com ele ficam perto da CNN 9 na média (menos de 2 pontos acima), com alguma melhora na precisão do tiro. Pela regra, a CNN 9 continua sendo a final.
- **O achado mais forte é a coluna do MP3:** a CNN 8, sem os microfones simulados, desaba com áudio comprimido e passa a chamar quase tudo de tiro; a CNN 9 se mantém. A robustez que os microfones simulados trouxeram vale também para uma distorção que a rede nunca viu, não só para o filtro com que treinou.
- **Eco e falhas de conexão** derrubam todos os modelos em 10 a 15 pontos: são riscos reais de uso que nenhum aumento de dados testado resolveu.

**Conclusão:** testamos o RawBoost no lugar dos microfones simulados (rodada 7), junto com eles (rodada 8) e em três condições adversas novas, e ficamos com a CNN 9, porque ele não trouxe melhora relevante: ficou um pouco abaixo na validação e menos de 2 pontos acima na média das condições adversas, dentro da variação entre sementes.

### 11.10 As curvas: o caminho da CNN 1 à final · Seção 4

In [ ]:
principais = ["CNN 1", "CNN 2 (dropout)", "CNN 3 (sem atalho)", "CNN 4 (sem atalho + delta)", escolha]
cores = ["#1565C0", "#2E7D32", "#C62828", "#6A1B9A", "#EF6C00"]
fig, (e1, e2) = plt.subplots(1, 2, figsize=(14, 4))
for nome, cor in zip(principais, cores):
    h = historicos[nome]; ep = range(1, len(h["loss"]) + 1)
    e1.plot(ep, h["val_loss"], "-", color=cor, label=nome); e1.plot(ep, h["loss"], "--", color=cor, alpha=0.4)
    e2.plot(ep, h["val_accuracy"], "-", color=cor, label=nome); e2.plot(ep, h["accuracy"], "--", color=cor, alpha=0.4)
e1.set_title("Perda (cheia: validação; tracejada: treino)"); e2.set_title("Acurácia (cheia: validação; tracejada: treino)")
for e in (e1, e2):
    e.set_xlabel("época"); e.legend(fontsize=8)
plt.tight_layout(); plt.show()
print("Atenção: CNN 1 e 2 validam no fold 9 original; as demais, no fold 9 sem atalho.")

### 11.11 O gráfico obrigatório: duas configurações lado a lado · Seção 4

A CNN 5 (vencedora da rodada 1, taxa 0,001) contra a rede final, as duas com a semente 42.

In [ ]:
par = [ref1, escolha] if escolha != ref1 else ["CNN 4 (sem atalho + delta)", escolha]
fig, (e1, e2) = plt.subplots(1, 2, figsize=(14, 4))
for nome, cor in zip(par, ["#1565C0", "#EF6C00"]):
    h = historicos[nome]; ep = range(1, len(h["loss"]) + 1)
    e1.plot(ep, h["val_loss"], "-", color=cor, label=f"{nome}: validação")
    e1.plot(ep, h["loss"], "--", color=cor, alpha=0.6, label=f"{nome}: treino")
    e2.plot(ep, h["val_accuracy"], "-", color=cor, label=f"{nome}: validação")
    e2.plot(ep, h["accuracy"], "--", color=cor, alpha=0.6, label=f"{nome}: treino")
e1.set_title("Perda"); e2.set_title("Acurácia (validação sem atalho)")
for e in (e1, e2):
    e.set_xlabel("época"); e.legend(fontsize=8)
plt.tight_layout(); plt.show()
for nome in par:
    h = historicos[nome]
    print(f"{nome}: melhor época {int(np.argmax(h['val_accuracy'])) + 1} de {len(h['loss'])}; "
          f"acurácia de validação nas últimas 10 épocas: {np.mean(h['val_accuracy'][-10:]):.1%} em média; "
          f"treino na última época: {h['accuracy'][-1]:.1%}")

**O que olhar:** com taxa 0,001, a acurácia de validação sobe rápido e oscila; a melhor época pode ser um **pico isolado**, que o `EarlyStopping` guarda, e o treino dispara para perto de 100% (sobreajuste). Na rede final (taxa menor, máscaras e misturas, que tornam o treino mais difícil), a subida é mais lenta e o treino fica mais perto da validação. A média das últimas épocas mostra isso melhor que o pico.

**A escolha, antes do teste:** escolhemos a CNN 9. Nas rodadas 1 a 3, a acurácia de validação parou de subir: as últimas mudanças ficaram dentro da variação entre sementes. Como o objetivo é o alerta de tiro, e na rua o tiro chega misturado a outros sons, a rodada 4 desempatou pelo recall do tiro com outro som (CNN 8). A rodada 5 mostrou, na validação, que a CNN 8 tinha ficado frágil a microfones piores, e dois aumentos de dados (misturas de sons sem tiro e microfones simulados) passaram a regra declarada antes; a combinação dos dois virou a CNN 9. A transferência com a YAMNet (rodada 6) não passou, nem o RawBoost no lugar dos microfones (rodada 7). Junto com os microfones (rodada 8), o RawBoost piorou o que estava validado e também ficou de fora. Todas as regras foram escritas antes de olhar o teste nesta versão.

### 11.12 O corte do tiro, também na validação · Seção 6

Pela regra da softmax, a rede só diz "tiro" quando o tiro é a classe **mais provável**. Numa triagem, deixar passar um tiro é pior que um alarme a mais. Como no corte do projeto de ML, podemos **baixar o limiar**: emitir um alerta quando a probabilidade de tiro passar de um valor menor.

Escolhemos o limiar **na validação sem atalho**, com uma regra escrita antes: **o maior recall do tiro com precisão de pelo menos 70%** (no máximo 3 alarmes falsos a cada 10 alertas). Se nenhum limiar chegar a 70%, fica o 0,5 (praticamente a própria resposta da softmax); nesse caso, a meta de precisão não foi atingida.

In [ ]:
X_va_final = montar(XB[valid], stats_final)
p_tiro_valid = prever(modelo_final, X_va_final)[:, TIRO]
tiro_real = (y_valid == TIRO)

linhas = []
for limiar in [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]:
    alerta = p_tiro_valid >= limiar
    acertos = (alerta & tiro_real).sum()
    linhas.append({"limiar": limiar, "alertas": int(alerta.sum()),
                   f"tiros achados (de {tiro_real.sum()})": int(acertos),
                   "recall (%)": round(100 * acertos / tiro_real.sum(), 1),
                   "precisão (%)": round(100 * acertos / max(alerta.sum(), 1), 1),
                   "alarmes falsos": int((alerta & ~tiro_real).sum())})
tabela_corte = pd.DataFrame(linhas).set_index("limiar")
display(tabela_corte)

aceitaveis = tabela_corte[tabela_corte["precisão (%)"] >= 70]
LIMIAR_TIRO = float(aceitaveis["recall (%)"].idxmax()) if len(aceitaveis) else 0.5
print("Limiar escolhido para o alerta de tiro:", LIMIAR_TIRO,
      "" if len(aceitaveis) else "(nenhum limiar chegou a 70% de precisão na validação; fica o 0,5)")

## Etapa 12 — O teste, uma vez · Seção 3

Agora, e só agora, o fold 10. Avaliamos o baseline e o modelo escolhido em três condições:
- **original:** o fold 10 como os notebooks públicos usam (com silêncio);
- **sem atalho:** som em posição sorteada e ruído fraco (20 dB abaixo do som), como na validação usada para escolher;
- **ruído forte:** o mesmo, com o ruído só 10 dB abaixo do som, como numa rua movimentada.

In [ ]:
def versao_teste_sem_atalho(snr_db):
    rng = np.random.default_rng(1234)
    espectros = [espectrograma(versao_sem_atalho(onda[:n], rng, snr_db)) for onda, n in zip(ondas_teste, n_teste)]
    return np.array(espectros, dtype="float32")

XB_teste_forte = versao_teste_sem_atalho(10)
condicoes = {"original": XA[teste], "sem atalho": XB[teste], "ruído forte": XB_teste_forte}

prev_base_teste = baseline.predict(np.zeros((len(y_teste), 1)))
linhas = [{"modelo": "Baseline (classe mais frequente)", "condição": "qualquer", **metricas(y_teste, prev_base_teste)},
          {"modelo": "Baseline (Random Forest)", "condição": "sem atalho",
           **metricas(y_teste, floresta.predict(resumo_faixas(XB[teste])))}]
prob_teste = {}
for nome_cond, espectros in condicoes.items():
    prob_teste[nome_cond] = prever(modelo_final, montar(espectros, stats_final))
    linhas.append({"modelo": escolha, "condição": nome_cond, **metricas(y_teste, prob_teste[nome_cond].argmax(1))})

# O alerta de tiro com o limiar escolhido na validação
alerta = prob_teste["sem atalho"][:, TIRO] >= LIMIAR_TIRO
tiro_real_t = (y_teste == TIRO)
linhas.append({"modelo": f"{escolha} + alerta (limiar {LIMIAR_TIRO})", "condição": "sem atalho",
               "acurácia (%)": float("nan"),
               "recall tiro (%)": round(100 * (alerta & tiro_real_t).sum() / tiro_real_t.sum(), 1),
               "precisão tiro (%)": round(100 * (alerta & tiro_real_t).sum() / max(alerta.sum(), 1), 1)})
# Tiro com outro som junto: as misturas do fold 10 (Etapa 4.2)
p_mist_teste = prever(modelo_final, montar(XM[OM[:, 4] == 10], stats_final))
linhas.append({"modelo": escolha, "condição": "tiro com outro som (só tiros)",
               "recall tiro (%)": round(100 * float((p_mist_teste.argmax(1) == TIRO).mean()), 1)})
linhas.append({"modelo": f"{escolha} + alerta (limiar {LIMIAR_TIRO})", "condição": "tiro com outro som (só tiros)",
               "recall tiro (%)": round(100 * float((p_mist_teste[:, TIRO] >= LIMIAR_TIRO).mean()), 1)})
tabela_teste = pd.DataFrame(linhas).set_index(["modelo", "condição"])
display(tabela_teste)

# Intervalo de confiança de 95% (método de Wilson): com só 32 tiros no teste, quanto esses números podem variar?
def intervalo(acertos, total, z=1.96):
    if total == 0:
        return (float("nan"), float("nan"))
    p = acertos / total
    centro = (p + z ** 2 / (2 * total)) / (1 + z ** 2 / total)
    margem = z * np.sqrt(p * (1 - p) / total + z ** 2 / (4 * total ** 2)) / (1 + z ** 2 / total)
    return (100 * (centro - margem), 100 * (centro + margem))

prev_t = prob_teste["sem atalho"].argmax(1)
for nome, disse in [("rede (classe mais provável)", prev_t == TIRO), (f"alerta (limiar {LIMIAR_TIRO})", alerta)]:
    a = int((disse & tiro_real_t).sum())
    r_lo, r_hi = intervalo(a, int(tiro_real_t.sum())); p_lo, p_hi = intervalo(a, int(disse.sum()))
    print(f"{nome}: recall {a}/{tiro_real_t.sum()} (IC95% {r_lo:.0f} a {r_hi:.0f}%) | "
          f"precisão {a}/{disse.sum()} (IC95% {p_lo:.0f} a {p_hi:.0f}%)")
acc_lo, acc_hi = intervalo(int((prev_t == y_teste).sum()), len(y_teste))
print(f"acurácia: IC95% {acc_lo:.1f} a {acc_hi:.1f}%")

**Esta é a tabela de avaliação final da Seção 3.** A linha "tiro com outro som" só tem recall: as misturas são todas tiros, então não há como medir alarmes falsos nelas.

**Intervalos de confiança:** com ~32 tiros, "100% de recall" quer dizer "acima de ~89% com 95% de confiança".

Se o número do teste ficar um pouco abaixo do da validação, é esperado: o modelo foi escolhido por ser o melhor **na validação**. Se ficar muito diferente, os folds são bem diferentes entre si (Seção 5).

## Etapa 13 — Onde a rede erra · Seção 5

Daqui em diante, analisamos o teste **sem atalho**, a condição mais parecida com o uso.

> ⚠️ **Daqui em diante, nada muda no modelo:** o teste só é usado para **medir e entender os erros**. Nenhuma das análises das Etapas 13 e 14 (matriz de confusão, certeza, Grad-CAM, intervenção, microfone, validação cruzada) escolhe ou ajusta nada. Uma ideia que surja delas só pode ser adotada se for decidida de novo **na validação**, com a regra escrita antes: foi o que fizemos com as bordas suavizadas e os microfones simulados, que surgiram destas análises numa versão anterior do notebook e foram decididos na rodada 5 (Etapa 11.6).

### 13.1 Precisão e recall de cada classe
- **Recall:** dos trechos que eram desta classe, quantos a rede acertou.
- **Precisão:** dos trechos que a rede chamou desta classe, quantos eram mesmo.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
prob_final = prob_teste["sem atalho"]
prev_final = prob_final.argmax(axis=1)
print(classification_report(y_teste, prev_final, labels=range(10), target_names=ROTULOS,
                            digits=2, zero_division=0))

### 13.2 Matriz de confusão: quem é confundido com quem

Linha: a classe certa. Coluna: o que a rede disse. A diagonal (em cinza) são os acertos; a cor vermelha mostra só os erros, para eles saltarem aos olhos.

In [ ]:
matriz = confusion_matrix(y_teste, prev_final, labels=range(10))
erros_m = matriz.copy(); np.fill_diagonal(erros_m, 0)          # a cor mostra só os erros
plt.figure(figsize=(9, 7.5))
plt.imshow(erros_m, cmap="Reds")
for i in range(10):
    for j in range(10):
        plt.text(j, i, matriz[i, j], ha="center", va="center", fontsize=9,
                 color="gray" if i == j else ("white" if erros_m[i, j] > erros_m.max() / 2 else "black"))
plt.xticks(range(10), ROTULOS, rotation=45, ha="right"); plt.yticks(range(10), ROTULOS)
plt.xlabel("o que a rede disse"); plt.ylabel("classe certa")
plt.title("Matriz de confusão no teste sem atalho (cor: só os erros; diagonal em cinza: acertos)")
plt.tight_layout(); plt.show()

pares = [(matriz[i, j], ROTULOS[i], ROTULOS[j]) for i in range(10) for j in range(10) if i != j and matriz[i, j] > 0]
print("As confusões mais frequentes:")
for n, certo, disse in sorted(pares, reverse=True)[:8]:
    print(f"  {n:3d} trechos de {certo:22s} → classificados como {disse}")

### 13.3 Trechos curtos × de 4 s, primeiro plano × fundo

In [ ]:
info_teste = meta.loc[teste, ["duracao", "salience"]].reset_index(drop=True)
acertou = (prev_final == y_teste)
grupos = pd.DataFrame({
    "duração": np.where(info_teste["duracao"] < 3.99, "curto (< 4 s)", "4 s completos"),
    "saliência": np.where(info_teste["salience"] == 1, "primeiro plano", "ao fundo"),
    "acertou": acertou})
for coluna in ["duração", "saliência"]:
    display(grupos.groupby(coluna)["acertou"].agg(trechos="size", acurácia=lambda a: round(100 * a.mean(), 1)))

### 13.4 Exemplos de erros

Os erros em que a rede estava **mais segura** (maior probabilidade na classe errada). Errar com muita certeza é o que mais custa na perda (Aula 3) e o que mais engana quem usa o modelo.

In [ ]:
certeza = prob_final.max(axis=1)
errados = np.where(~acertou)[0]
piores = errados[np.argsort(-certeza[errados])][:6]

if len(piores):
    fig, eixos = plt.subplots(2, 3, figsize=(15, 6))
    for eixo, k in zip(eixos.flat, piores):
        eixo.imshow(XB[teste][k].astype("float32"), origin="lower", aspect="auto", cmap="magma")
        eixo.set_title(f"certo: {ROTULOS[y_teste[k]]}\nrede: {ROTULOS[prev_final[k]]} ({certeza[k]:.0%})", fontsize=10)
        eixo.set_xticks([]); eixo.set_yticks([])
    plt.tight_layout(); plt.show()
    for k in piores[:3]:
        print(f"certo: {CLASSES[y_teste[k]]} | rede: {CLASSES[prev_final[k]]} ({certeza[k]:.0%})")
        display(Audio(ondas_teste[k][:n_teste[k]], rate=SR))

**Os erros que mais importam para o caso de uso:** os tiros que a rede **não** reconheceu e os trechos que ela chamou de tiro sem ser (os de maior probabilidade de tiro primeiro).

In [ ]:
def mostrar_trechos(indices, titulo):
    if len(indices) == 0:
        print(titulo + ": nenhum"); return
    fig, eixos = plt.subplots(1, len(indices), figsize=(3.2 * len(indices), 2.9))
    for e, k in zip(np.atleast_1d(eixos), indices):
        e.imshow(XB[teste][k].astype("float32"), origin="lower", aspect="auto", cmap="magma")
        e.set_title(f"certo: {ROTULOS[y_teste[k]]}\nrede: {ROTULOS[prev_final[k]]}\np(tiro) = {prob_final[k, TIRO]:.0%}", fontsize=8)
        e.set_xticks([]); e.set_yticks([])
    plt.suptitle(titulo, y=1.1); plt.show()

tiros_perdidos = np.where((y_teste == TIRO) & (prev_final != TIRO))[0]
alarmes_falsos = np.where((y_teste != TIRO) & (prev_final == TIRO))[0]
alarmes_falsos = alarmes_falsos[np.argsort(-prob_final[alarmes_falsos, TIRO])]
print(f"Tiros no teste: {(y_teste == TIRO).sum()} | não reconhecidos: {len(tiros_perdidos)} | "
      f"alarmes falsos de tiro: {len(alarmes_falsos)}")
mostrar_trechos(tiros_perdidos[:5], "Tiros que a rede não reconheceu")
mostrar_trechos(alarmes_falsos[:5], "Alarmes falsos de tiro (os mais confiantes)")
for k in alarmes_falsos[:2]:
    print(f"alarme falso: era {ROTULOS[y_teste[k]]}, p(tiro) = {prob_final[k, TIRO]:.0%}")
    display(Audio(ondas_teste[k][:n_teste[k]], rate=SR))

### 13.5 Quando a rede tem pouca certeza

Como no passo 10.2 da Aula 4: casos em que a maior probabilidade fica baixa poderiam ir para **conferência humana**. A tabela por faixa de certeza mostra quanto a rede acerta em cada nível de confiança: a certeza separa bem acertos de erros. Ela **não** escolhe um corte: um limite para mandar casos à conferência humana teria de ser escolhido na **validação** e só depois conferido num teste novo (o teste daqui já foi visto).

In [ ]:
LIMIAR_CERTEZA = 0.6
pouca = certeza < LIMIAR_CERTEZA
print(f"Trechos com certeza abaixo de {LIMIAR_CERTEZA:.0%}: {pouca.sum()} de {len(certeza)} ({pouca.mean():.0%})")
print(f"Acurácia quando a rede está segura: {acertou[~pouca].mean():.0%}")
if pouca.any():
    print(f"Acurácia quando tem pouca certeza:  {acertou[pouca].mean():.0%}")

plt.figure(figsize=(8, 3.5))
plt.hist(certeza[acertou], bins=20, alpha=0.7, density=True, label="acertos")
plt.hist(certeza[~acertou], bins=20, alpha=0.7, density=True, label="erros")
plt.axvline(LIMIAR_CERTEZA, ls=":", color="gray", label=f"limiar {LIMIAR_CERTEZA}")
plt.xlabel("probabilidade da classe prevista (certeza)"); plt.ylabel("densidade"); plt.legend()
plt.title("Certeza da rede nos acertos e nos erros (teste sem atalho)")
plt.show()
print("Erros com certeza acima de 90%:", int((~acertou & (certeza > 0.9)).sum()))

# A certeza separa acertos de erros? Acurácia por faixa de certeza
faixas = pd.cut(certeza, [0, 0.5, 0.7, 0.9, 1.0], labels=["abaixo de 50%", "50 a 70%", "70 a 90%", "90% ou mais"])
tabela_certeza = pd.DataFrame({"faixa de certeza": faixas, "acertou": acertou}).groupby("faixa de certeza", observed=True)["acertou"].agg(
    trechos="size", acurácia=lambda a: round(100 * a.mean(), 1))
display(tabela_certeza)

# Foco nas classes de segurança: para onde vão os perdidos e de onde vêm os alarmes falsos
foco = []
for nome in ["gun_shot", "siren"]:
    k = CLASSES.index(nome); real, disse = (y_teste == k), (prev_final == k)
    foco.append({"classe": nome, "acertos": int((real & disse).sum()),
                 "não detectados": int((real & ~disse).sum()), "alarmes falsos": int((~real & disse).sum()),
                 "não detectados foram chamados de": pd.Series([CLASSES[c] for c in prev_final[real & ~disse]]).value_counts().to_dict(),
                 "alarmes falsos eram, na verdade": pd.Series([CLASSES[c] for c in y_teste[~real & disse]]).value_counts().to_dict()})
tabela_foco = pd.DataFrame(foco).set_index("classe")
tabela_foco

### 13.6 Validação cruzada: o fold 10 representa a base? · Seções 3 e 5

Até aqui, tudo dependeu de **um** fold de validação (9) e **um** de teste (10). Os autores da base recomendam repetir com cada um dos 10 folds como teste. Fazemos isso **só com a receita escolhida**, sem mudar nada nela: não é uma nova escolha de modelo, é uma medida de **quanto o resultado depende do fold**.

Para cada fold de teste *K*: validação no fold anterior (para *K* = 1, o fold 10), treino nos outros 8. A padronização usa só os 8 folds de treino de cada rodada.

**Demora:** 10 redes, cerca de 20 a 30 min com GPU. Fica **desligada** por padrão; os resultados de quem já rodou ficam no Drive e são carregados.

In [ ]:
RODAR_VALIDACAO_CRUZADA = False      # mude para True para treinar as redes que ainda não estão no cache

r_final = receitas[escolha]; curto = escolha.split()[1]
linhas_cv, prev_cv, real_cv, recall_cv = [], [], [], {}
for K in range(1, 11):
    V = 10 if K == 1 else K - 1
    tr, va, te = (~meta["fold"].isin([K, V])).values, (meta["fold"] == V).values, (meta["fold"] == K).values
    ytr, yva, yte = (meta.loc[m, "classID"].values for m in (tr, va, te))
    nome = f"cv_{curto}_teste{K}"
    em_cache = os.path.exists(os.path.join(PASTA_MODELOS, f"{nome}_{VERSAO_TREINO}.weights.h5"))
    if not (em_cache or RODAR_VALIDACAO_CRUZADA):
        continue
    s = estatisticas(XB[tr], canais=r_final["canais"])
    modelo = criar_cnn(nome, dropout=r_final["dropout"], canais=r_final["canais"], l2=r_final["l2"],
                       aumento=r_final["aumento"], desloc_freq=r_final.get("desloc_freq", 0.0),
                       mascaras=r_final.get("mascaras", False), semente=42)    # a receita escolhida, inteira
    precisa_treinar = (not em_cache) or RETREINAR
    X_tr, X_va = None, None
    if precisa_treinar:     # a receita inteira, com os aumentos de dados só dos folds de treino desta rodada
        (X_tr, ytr), X_va = dados_de_treino(r_final, tr, s), montar(XB[va], s)
    h = treinar(modelo, X_tr, X_va, reduzir_lr=r_final["reduzir_lr"], taxa=r_final.get("taxa", 0.001),
                lote=r_final.get("lote", 32), y_tr=ytr, y_va=yva)
    del X_tr, X_va; gc.collect()
    p = prever(modelo, montar(XB[te], s)).argmax(1)
    p_orig = prever(modelo, montar(XA[te], s)).argmax(1)
    p_mist = prever(modelo, montar(XM[OM[:, 4] == K], s)).argmax(1)          # tiros com outro som, fold K
    m = metricas(yte, p)
    linhas_cv.append({"fold de teste": K, "fold de validação": V, "trechos": int(te.sum()), "tiros": int((yte == TIRO).sum()),
                      "acurácia validação (%)": round(100 * max(h["val_accuracy"]), 1),
                      "acurácia teste sem atalho (%)": m["acurácia (%)"],
                      "acurácia teste original (%)": metricas(yte, p_orig)["acurácia (%)"],
                      "F1-macro teste sem atalho (%)": m["F1-macro (%)"],
                      "baseline (%)": round(float(100 * (yte == np.bincount(meta.loc[tr, "classID"]).argmax()).mean()), 1),
                      "recall tiro (%)": m["recall tiro (%)"], "precisão tiro (%)": m["precisão tiro (%)"],
                      "tiro com outro som (%)": round(100 * float((p_mist == TIRO).mean()), 1)})
    recall_cv[K] = {c: round(float(100 * (p[yte == i] == i).mean()), 1) for i, c in enumerate(CLASSES)}
    prev_cv.append(p); real_cv.append(yte)

if linhas_cv:
    tabela_cv = pd.DataFrame(linhas_cv).set_index("fold de teste")
    colunas = ["acurácia validação (%)", "acurácia teste sem atalho (%)", "acurácia teste original (%)", "F1-macro teste sem atalho (%)",
               "baseline (%)", "recall tiro (%)", "precisão tiro (%)", "tiro com outro som (%)"]
    resumo_cv = tabela_cv[colunas].agg(["mean", "std", "min", "max"]).round(1)
    resumo_cv.index = ["média", "desvio", "menor", "maior"]
    display(tabela_cv); display(resumo_cv)
    if 10 in tabela_cv.index:
        v9 = tabela_cv.loc[10, "acurácia validação (%)"]           # quando o teste é o 10, a validação é o fold 9
        media_v = tabela_cv["acurácia validação (%)"].mean()
        print(f"O fold 9, onde escolhemos tudo, deu {v9:.1f}% como validação; a média dos 10 folds de validação é "
              f"{media_v:.1f}%. {'O fold 9 é mais fácil que a média: a validação do projeto sai otimista.' if v9 > media_v + 2 else ''}")
else:
    print("Validação cruzada desligada e sem resultados no cache. Mude RODAR_VALIDACAO_CRUZADA para True.")

In [ ]:
if linhas_cv:
    folds = list(tabela_cv.index)
    accs = tabela_cv["acurácia teste sem atalho (%)"]
    fig, (e1, e2) = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1, 1.3]})
    e1.bar(folds, accs, color=["#EF6C00" if k == 10 else "#1565C0" for k in folds])
    e1.bar(folds, tabela_cv["baseline (%)"], color="gray", label="baseline")
    e1.axhline(accs.mean(), ls="--", color="black", label=f"média {accs.mean():.1f}%")
    for k, a in zip(folds, accs):
        e1.text(k, a + 1, f"{a:.0f}", ha="center", fontsize=8)
    e1.set_xticks(folds); e1.set_ylim(0, 95); e1.legend(loc="lower right", fontsize=8)
    e1.set_xlabel("fold de teste (laranja: o fold de teste do projeto)"); e1.set_ylabel("acurácia (%)")
    e1.set_title(f"{escolha}: cada fold como teste")
    rc = pd.DataFrame(recall_cv); ordem = rc.mean(axis=1).sort_values().index
    e2.boxplot([rc.loc[c] for c in ordem], orientation="horizontal", tick_labels=[NOMES_PT[c] for c in ordem])
    if 10 in rc.columns:
        e2.plot(rc.loc[ordem, 10], range(1, len(ordem) + 1), "o", color="#EF6C00", label="fold 10")
        e2.legend(fontsize=8)
    e2.set_xlabel("recall nos folds (%)"); e2.set_title("Quanto cada classe varia de um fold para outro")
    plt.tight_layout(); plt.show()

    # As confusões mais frequentes, somando todos os folds
    m = confusion_matrix(np.concatenate(real_cv), np.concatenate(prev_cv), labels=range(10))
    total = m.sum(axis=1); np.fill_diagonal(m, 0)
    print(f"Acurácia somando todos os trechos: {100 * (np.concatenate(real_cv) == np.concatenate(prev_cv)).mean():.1f}%")
    print("Confusões mais frequentes nos folds avaliados:")
    for i, j in zip(*np.unravel_index(np.argsort(-m.ravel())[:6], m.shape)):
        print(f"  {m[i, j]:4d} trechos de {CLASSES[i]:17s} → {CLASSES[j]:17s} ({100 * m[i, j] / total[i]:.0f}% da classe)")

**Leitura:**
- **O número do fold 10 sozinho pode enganar.** Se ele ficar acima da média dos 10 folds, o fold 10 é mais fácil que a base como um todo, e a estimativa honesta para o uso é a **média ± desvio** da validação cruzada.
- **O fold 9 também.** Escolhemos todas as configurações nele. Se ele for um fold fácil, as escolhas foram feitas num terreno favorável, e o número da validação também sai otimista.
- **Quais classes são estáveis?** Uma caixa estreita no gráfico da direita é uma classe que a rede reconhece bem em qualquer fold. Uma caixa larga depende de quais gravações caíram no teste.
- **As confusões somadas** mostram os grupos de sons parecidos, com muito mais trechos que um fold só.

### 13.7 Mudança de distribuição: e se o microfone for outro? · Seções 5 e 6

A base foi gravada com microfones bons. Um sensor de rua, uma câmera de segurança ou um celular gravam pior. Isso é **mudança de distribuição** (Aula 5): os dados do uso real não se parecem com os do treino. Simulamos três microfones piores no fold 10 (versão sem atalho), **sem treinar nada de novo**:

| Microfone simulado | O que fazemos com o áudio |
|---|---|
| telefone | filtro que só deixa passar de 300 a 3.400 Hz (a faixa de uma ligação) |
| microfone barato | reduzimos a taxa de amostragem para 8.000 Hz e voltamos: some tudo acima de 4.000 Hz |
| saturado | cortamos a onda em 20% do pico: o som "estoura", como um tiro perto de um microfone sensível |

In [ ]:
from scipy.signal import butter, sosfilt

def telefone(y):
    return sosfilt(butter(4, [300, 3400], btype="bandpass", fs=SR, output="sos"), y).astype("float32")

def microfone_barato(y):
    y8 = librosa.resample(y, orig_sr=SR, target_sr=8000)
    return librosa.util.fix_length(librosa.resample(y8, orig_sr=8000, target_sr=SR), size=N_AMOSTRAS)

def saturado(y):
    limite = 0.2 * np.abs(y).max()
    return np.clip(y, -limite, limite)

def medir_alerta(p, y_real):
    alerta = p[:, TIRO] >= LIMIAR_TIRO; real = y_real == TIRO
    return {"alerta: recall tiro (%)": round(100 * float((alerta & real).sum() / real.sum()), 1),
            "alerta: precisão tiro (%)": round(100 * float((alerta & real).sum() / max(alerta.sum(), 1)), 1)}

linhas_mic = [{"microfone": "como na base", **metricas(y_teste, prob_teste["sem atalho"].argmax(1)),
               **medir_alerta(prob_teste["sem atalho"], y_teste)}]
for nome, transformar in [("telefone (300 a 3.400 Hz)", telefone), ("microfone barato (8.000 Hz)", microfone_barato),
                          ("saturado (pico cortado em 20%)", saturado)]:
    rng = np.random.default_rng(1234)
    espectros = np.array([espectrograma(transformar(versao_sem_atalho(onda[:n], rng))) for onda, n in zip(ondas_teste, n_teste)])
    p = prever(modelo_final, montar(espectros, stats_final))
    linhas_mic.append({"microfone": nome, **metricas(y_teste, p.argmax(1)), **medir_alerta(p, y_teste)})
tabela_microfone = pd.DataFrame(linhas_mic).set_index("microfone")
tabela_microfone

**Leitura:**
- **Telefone e microfone barato** tiram as frequências altas. A CNN 8 caía para perto de 30% aqui e chamava quase tudo de tiro; por isso a CNN 9 treinou com **microfones simulados** (rodada 5, Etapa 11.6). Cuidado ao comparar: os filtros do treino são parecidos com estes, então esta tabela é otimista para a CNN 9. Um microfone real de verdade ainda precisa ser testado antes de ir para a rua.
- **Saturado:** quase todo tiro gravado perto já vem estourado na base (Etapa 5), então é a mudança que menos deve afetar o tiro; para as outras classes, vira um som novo.
- Nenhuma dessas condições foi usada para escolher nada: são medidas do risco de uso.

### 13.8 Grad-CAM: onde estão as pistas para cada classe? · Aula 5

O **Grad-CAM** (Aula 5, slides 53 e 54) combina os mapas da última `Conv2D` com o gradiente da pontuação de uma classe (o *logit*, antes da softmax). O resultado é um mapa de calor sobre o espectrograma: as regiões que mais **aumentam** a pontuação daquela classe. É uma pista para formular hipóteses, **não** uma prova de causa (slide 58).

Mostramos, para cada caso, o espectrograma, o mapa da classe que a rede disse e o mapa da classe certa:
1. um tiro sozinho, acertado;
2. um tiro somado a outro som (misturas da Etapa 4.2), acertado;
3. o alarme falso de tiro mais confiante;
4. uma britadeira classificada como motor (a maior confusão do teste).

A última `Conv2D` tem resolução menor que a entrada; o mapa é ampliado para 64 × 173, então os contornos são aproximados.

In [ ]:
import tensorflow as tf

k_tiro = int(np.where((y_teste == TIRO) & (prev_final == TIRO))[0][0])   # um tiro do teste que a rede acertou
x_tiro = montar(XB[teste][k_tiro:k_tiro + 1], stats_final)[0]

def gradcam(modelo, x, classe):
    # Grad-CAM na última Conv2D, para o logit da classe pedida
    i_ultima_conv = max(i for i, c in enumerate(modelo.layers) if isinstance(c, layers.Conv2D))
    with tf.GradientTape() as fita:
        h = tf.convert_to_tensor(x[np.newaxis])
        for i, camada in enumerate(modelo.layers):
            if isinstance(camada, layers.Dense):             # a saída: usamos o logit, antes da softmax
                logit = (tf.matmul(h, camada.kernel) + camada.bias)[0, classe]
            else:
                h = camada(h, training=False)
            if i == i_ultima_conv:
                mapas = h
                fita.watch(mapas)
    gradientes = fita.gradient(logit, mapas)[0]                 # (altura, largura, filtros)
    pesos = tf.reduce_mean(gradientes, axis=(0, 1))              # importância de cada filtro
    mapa = tf.nn.relu(tf.reduce_sum(mapas[0] * pesos, axis=-1))  # só o que aumenta a pontuação
    mapa = mapa / (tf.reduce_max(mapa) + 1e-8)
    return tf.image.resize(mapa[..., tf.newaxis], (N_MELS, N_QUADROS)).numpy()[..., 0]

def mostrar_gradcam(espectro, x, classe_certa, titulo):
    p = prever(modelo_final, x[np.newaxis])[0]; disse = int(p.argmax())
    fig, eixos = plt.subplots(1, 3, figsize=(15, 3))
    eixos[0].imshow(espectro, origin="lower", aspect="auto", cmap="magma")
    eixos[0].set_title(f"{titulo}\ncerto: {ROTULOS[classe_certa]} | rede: {ROTULOS[disse]} ({p[disse]:.0%})", fontsize=9)
    for e, classe, nome in [(eixos[1], disse, "mapa da classe que a rede disse"), (eixos[2], classe_certa, "mapa da classe certa")]:
        e.imshow(espectro, origin="lower", aspect="auto", cmap="gray")
        e.imshow(gradcam(modelo_final, x, classe), origin="lower", aspect="auto", cmap="jet", alpha=0.45, vmin=0, vmax=1)
        e.set_title(f"{nome}: {ROTULOS[classe]}", fontsize=9)
    for e in eixos:
        e.set_xlabel("tempo"); e.set_ylabel("frequência (mel)"); e.set_xticks([]); e.set_yticks([])
    plt.tight_layout(); plt.show()

# 1. Tiro sozinho
mostrar_gradcam(XB[teste][k_tiro].astype("float32"), x_tiro, TIRO, "1. Tiro sozinho")

# 2. Tiro com outro som (misturas do fold 10, o tiro 6 dB acima)
idx_m = np.where((OM[:, 4] == 10) & (OM[:, 2] == 6))[0]
X_m = montar(XM[idx_m], stats_final)
p_m = prever(modelo_final, X_m)
j = int(np.where(p_m.argmax(1) == TIRO)[0][0]) if (p_m.argmax(1) == TIRO).any() else 0
mostrar_gradcam(XM[idx_m[j]].astype("float32"), X_m[j], TIRO, f"2. Tiro somado a {ROTULOS[OM[idx_m[j], 3]]}")

# 3. O alarme falso de tiro mais confiante
if len(alarmes_falsos):
    k = int(alarmes_falsos[0])
    mostrar_gradcam(XB[teste][k].astype("float32"), montar(XB[teste][k:k + 1], stats_final)[0], int(y_teste[k]),
                    "3. Alarme falso de tiro")

# 4. A maior confusão: britadeira classificada como motor
k_conf = np.where((y_teste == CLASSES.index("jackhammer")) & (prev_final == CLASSES.index("engine_idling")))[0]
if len(k_conf):
    k = int(k_conf[0])
    mostrar_gradcam(XB[teste][k].astype("float32"), montar(XB[teste][k:k + 1], stats_final)[0], int(y_teste[k]),
                    "4. Britadeira → motor")

**Como ler:**
- **Tiro sozinho:** o calor deve ficar sobre o estrondo (a faixa vertical forte) e não sobre o ruído de fundo ou as bordas vazias. Se ficar no ruído, é sinal de atalho.
- **Tiro com outro som:** a rede precisa achar o estrondo no meio do outro som. O calor no estrondo, e não no som de fundo, é o que se espera.
- **Alarme falso:** onde o calor cai? Se cai no **começo ou no fim abrupto** do trecho, a rede está tomando a borda cortada por estrondo. A Etapa 13.9 testa essa hipótese.
- **Britadeira → motor:** compare os dois mapas. Se os dois marcam as mesmas faixas graves contínuas, a rede não acha o que diferencia uma da outra.

> ⚠️ **Limite do Grad-CAM (Aula 5, slide 58):** o mapa não é probabilidade por pixel nem causa. Ele sugere hipóteses; para testá-las, trocamos só a pista suspeita e medimos de novo, como no experimento do quadrado colorido da aula.

### 13.9 Teste de intervenção: a borda cortada vira "tiro"? · Aula 5

Já fizemos um teste de intervenção na Etapa 12: o **original × sem atalho** troca só o silêncio do fim, e a precisão do tiro despenca na versão original. Aqui testamos uma segunda pista, sugerida pelas imagens dos alarmes falsos (Etapa 13.4) e pelo Grad-CAM.

**Hipótese:** um trecho curto de furadeira ou de sirene, cortado de uma gravação maior e colocado no meio do ruído (versão sem atalho), começa e termina **de forma abrupta**, como um estrondo. A rede pode estar tomando essa borda por tiro.

**Intervenção:** nos trechos curtos (menos de 4 s) do teste, suavizamos **só as bordas** (o volume sobe e desce gradualmente em 100 ms), com a mesma posição e o mesmo ruído. Se a hipótese estiver certa, os alarmes falsos caem e os tiros continuam sendo achados.

In [ ]:
duracao_teste = meta.loc[teste, "duracao"].values
curto = duracao_teste < 3.99
nao_tiro = y_teste != TIRO
print(f"Alarmes falsos de tiro no teste: {int(((prev_final == TIRO) & nao_tiro).sum())}")
print(f"  entre os não-tiros curtos: {100 * (prev_final[nao_tiro & curto] == TIRO).mean():.1f}% | "
      f"entre os não-tiros de 4 s: {100 * (prev_final[nao_tiro & ~curto] == TIRO).mean():.1f}%")

def suavizar_bordas(y, segundos=0.1):
    y = y.copy(); k = min(int(segundos * SR), len(y) // 2)
    rampa = np.linspace(0, 1, k, dtype="float32")
    y[:k] *= rampa; y[-k:] *= rampa[::-1]
    return y

linhas_int = []
idx_curtos = np.where(curto)[0]
for nome, transformar in [("bordas como estão", lambda y: y), ("bordas suavizadas (100 ms)", suavizar_bordas)]:
    espectros = np.array([espectrograma(versao_sem_atalho(transformar(ondas_teste[k][:n_teste[k]]), np.random.default_rng(k)))
                          for k in idx_curtos])
    p = prever(modelo_final, montar(espectros, stats_final)).argmax(1)
    yc = y_teste[idx_curtos]
    linhas_int.append({"trechos curtos do teste": nome,
                       "alarmes falsos (% dos não-tiros)": round(100 * float((p[yc != TIRO] == TIRO).mean()), 1),
                       "recall do tiro (%)": round(100 * float((p[yc == TIRO] == TIRO).mean()), 1),
                       "acurácia (%)": round(100 * float((p == yc).mean()), 1)})
tabela_intervencao = pd.DataFrame(linhas_int).set_index("trechos curtos do teste")
tabela_intervencao

**Leitura:**
- Se os alarmes falsos **caem** com as bordas suavizadas e o recall do tiro **não muda**, a borda cortada é parte da pista que a rede usa para dizer "tiro". É um atalho criado pela forma como a base foi recortada (e pela nossa versão sem atalho, que coloca o trecho no meio do ruído).
- Se caem só em parte, a borda explica parte dos alarmes falsos; o resto vem de sons realmente impulsivos (britadeira, latido).
- **Treinar com as bordas suavizadas** (para a rede não aprender que "começo seco" é tiro) foi testado na rodada 5 (Etapa 11.6) e **não passou** a regra: os alarmes falsos não caíram de forma consistente entre sementes, e os que sobram vêm sobretudo de ar-condicionado.

## Etapa 14 — Usar o modelo

### 14.1 Prever um áudio novo

O notebook precisa "gerar previsões" (descrição do projeto). A função faz com um áudio novo **o mesmo** pré-processamento do treino: mono, 22.050 Hz, 4 s, espectrograma de mel, padronização com as estatísticas do treino.

In [ ]:
def prever_trecho(onda, mostrar=3):
    y = onda[:N_AMOSTRAS]
    if len(y) < N_AMOSTRAS:                       # trecho curto: a mesma preparação do treino do modelo escolhido
        if escolha not in receitas:              # só as CNN 1 e 2 treinaram com a versão original
            y = versao_original(y)
        else:
            y = versao_sem_atalho(y, np.random.default_rng(0))
    p = prever(modelo_final, montar(espectrograma(y)[np.newaxis], stats_final))[0]
    for i in np.argsort(p)[::-1][:mostrar]:
        print(f"  {CLASSES[i]:17s} {p[i]:6.1%}")
    return p

k = int(np.where(y_teste == TIRO)[0][0])
print("Rótulo verdadeiro:", CLASSES[y_teste[k]])
prever_trecho(ondas_teste[k][:n_teste[k]])
display(Audio(ondas_teste[k][:n_teste[k]], rate=SR))

# Para testar um arquivo seu (.wav) localmente, descomentem:
# prever_trecho(carregar_bruto("meu_audio.wav"))
# Para testar um arquivo seu (.wav) no Colab, descomentem:
# from google.colab import files
# enviado = files.upload()
# prever_trecho(carregar_bruto(list(enviado)[0]))

### 14.2 Uma gravação longa, com vários sons em sequência

O modelo classifica **um** som por trecho. Para uma gravação longa, ele analisa a gravação em **janelas** de 4 s, andando de meio em meio segundo, e diz o som mais provável de cada janela. O resultado é uma linha do tempo.

Montamos uma gravação de exemplo juntando trechos do fold de teste (motor, tiro, latido, sirene), separados por 3 s de ruído fraco. Quando uma janela pega dois sons, a rede aponta o que domina; sons **sobrepostos** no mesmo instante continuam fora do alcance do modelo.

In [ ]:
rng = np.random.default_rng(7)
pedacos, rotulos_verdade, t = [], [], 0.0
for nome in ["engine_idling", "gun_shot", "dog_bark", "siren", "gun_shot"]:
    k = int(np.where(y_teste == CLASSES.index(nome))[0][rng.integers(0, 5)])
    trecho = ondas_teste[k][:n_teste[k]] / (np.abs(ondas_teste[k][:n_teste[k]]).max() + 1e-9) * 0.5
    pausa = np.zeros(int(3.0 * SR), dtype="float32")      # 3 s só de ruído de fundo entre os sons
    pedacos += [trecho, pausa]
    rotulos_verdade.append((t, t + len(trecho) / SR, nome)); t += (len(trecho) + len(pausa)) / SR
gravacao = np.concatenate(pedacos)
gravacao = gravacao + rng.normal(0, 0.01, len(gravacao)).astype("float32")   # ruído de rua fraco

passo = int(0.5 * SR)
inicios = np.arange(0, len(gravacao) - N_AMOSTRAS + 1, passo)
janelas = np.array([espectrograma(gravacao[i:i + N_AMOSTRAS]) for i in inicios])
p_janelas = prever(modelo_final, montar(janelas, stats_final))

fig, (e1, e2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
S = librosa.power_to_db(librosa.feature.melspectrogram(y=gravacao, sr=SR, hop_length=HOP, n_mels=N_MELS), ref=np.max)
e1.imshow(S, origin="lower", aspect="auto", cmap="magma", extent=[0, len(gravacao) / SR, 0, N_MELS])
for ini, fim, nome in rotulos_verdade:
    e1.axvspan(ini, fim, color="white", alpha=0.08); e1.text(ini + 0.05, N_MELS - 6, nome, color="white", fontsize=9)
e1.set_title("A gravação (em branco, o som que foi colocado em cada trecho)"); e1.set_ylabel("frequência (mel)")
centro = (inicios + N_AMOSTRAS / 2) / SR
e2.plot(centro, p_janelas[:, TIRO], "o-", color="#C62828", label="probabilidade de tiro")
e2.axhline(LIMIAR_TIRO, ls=":", color="gray", label=f"limiar do alerta ({LIMIAR_TIRO})")
e2.set_ylim(0, 1); e2.set_xlabel("tempo (s), centro da janela"); e2.legend(loc="upper right")
plt.tight_layout(); plt.show()

print("Linha do tempo (início da janela → som mais provável):")
anterior = None
for i, p in zip(inicios, p_janelas):
    classe = CLASSES[int(p.argmax())]
    if classe != anterior:
        print(f"  {i / SR:5.1f} s → {classe} ({p.max():.0%})"); anterior = classe

**Leitura:**
- A curva vermelha mostra onde um alerta de tiro seria emitido. Quando a janela pega o tiro junto com um som mais longo e mais forte, o som longo domina.
- **Nas janelas só com ruído, a rede ainda escolhe uma das 10 classes**, com pouca certeza. Ela não tem a opção "nenhum desses sons": é uma limitação importante para o uso real (Etapa 15).

In [ ]:
# Salvar o modelo escolhido (Aula 4, passo 13), junto com as estatísticas de padronização
modelo_final.save(os.path.join(PASTA_CACHE, "modelo_final.keras"))
json.dump({**stats_final, "escolha": escolha, "limiar_tiro": LIMIAR_TIRO},
          open(os.path.join(PASTA_CACHE, "modelo_final_config.json"), "w"))
print("Modelo salvo em", PASTA_CACHE)

# Conferência: o modelo carregado do arquivo dá o mesmo resultado no teste?
carregado = keras.models.load_model(os.path.join(PASTA_CACHE, "modelo_final.keras"))
acc_carregado = (prever(carregado, montar(XB[teste], stats_final)).argmax(1) == y_teste).mean()
acc_original = (prob_teste["sem atalho"].argmax(1) == y_teste).mean()
print(f"Acurácia no teste — modelo na memória: {acc_original:.1%} | carregado do arquivo: {acc_carregado:.1%}")

## Etapa 15 — Uso pretendido e restrições · Seção 6

**Decisão que o modelo pode apoiar:** numa central de monitoramento que recebe áudio de sensores na rua, **priorizar** janelas com alta probabilidade de tiro para um operador ouvir e decidir. O modelo filtra; a pessoa decide.

**Quem usaria:** operadores de centrais de monitoramento, com supervisão.

**Informação disponível no momento da previsão:** só o áudio. Nada de metadados como duração, saliência ou gravação de origem.

**Consequências dos erros:**
- **Falso negativo** (tiro não detectado): um disparo real passa despercebido. Grave, mas o modelo é um apoio, não o único meio de alerta.
- **Tiro com outro som junto:** a condição mais parecida com a rua. A Etapa 12 mede quantos desses tiros o modelo acha; mesmo com o treino da rodada 4, parte deles passa, sobretudo quando o outro som é tão forte quanto o tiro.
- **Falso positivo** (alarme falso): uma viatura deslocada à toa, e operadores que passam a ignorar os alertas. Fogos de artifício, escapamento de moto e bombinhas soam como tiro e **não existem na base**.

**Não deve ser usado, mesmo funcionando bem:**
- para **acionar ação policial sozinho**, sem conferência humana;
- como **prova** de que houve um disparo;
- em **outra cidade ou com outros microfones** sem avaliar antes com áudio de lá (a base é do Freesound, com muito som de Nova York);
- em ambientes com **sons fora das 10 classes**, ou sem som nenhum: a rede sempre escolhe uma das 10, mesmo para um som que nunca ouviu (Etapa 14.2); sons parecidos com tiro, como fogos de artifício, bombinhas e escapamento, não existem na base e precisariam entrar no treino;
- com **outro microfone** sem avaliar antes: a Etapa 13.7 mostra quanto o desempenho cai.

**O que verificar antes de um uso real:** gravar e rotular áudio do local de uso; medir recall e precisão do tiro nele; incluir fogos e escapamentos; manter o limiar do alerta e o limiar de certeza (Etapas 11.12 e 13.5) ajustados com dados do local.

In [ ]:
# Resumo dos números para o relatório
print("=== Atalhos (validação) ===");                         display(tabela_atalhos)
print("=== Comparação na validação (fold 9), rodada 1 ==="); display(tabela_valid)
print("=== Rodada 2 (3 sementes) ===");                         display(tabela_r2)
print("=== Rodada 3 (3 sementes) ===");                         display(tabela_r3)
print("=== Rodada 4 (3 sementes) ===");                         display(tabela_r4)
print("=== Diagnóstico da rodada 5 (validação) ===");          display(diagnostico)
print("=== Rodada 5 (3 sementes) ===");                         display(tabela_r5)
print("=== Rodada 6 · YAMNet (3 sementes) ===");                display(tabela_r6)
print("=== Rodada 7 · RawBoost no lugar (3 sementes) ===");     display(tabela_r7)
print("=== Rodada 8 · RawBoost junto (3 sementes) ===");        display(tabela_r8)
print("=== Condições adversas novas (validação) ===");         display(tabela_adversas)
print("=== Outros microfones (teste) ===");                     display(tabela_microfone)
print("=== Corte do tiro (validação) ===");                    display(tabela_corte)
print("=== Teste (fold 10), modelo escolhido:", escolha, "==="); display(tabela_teste)
if len(linhas_cv):
    print("=== Validação cruzada (10 folds) ==="); display(resumo_cv)

# Tabelas em CSV + figuras (já salvas) num .zip, para o relatório e os slides
import shutil
tabelas = {"atalhos": tabela_atalhos, "validacao_rodada1": tabela_valid, "rodada2": tabela_r2, "rodada3": tabela_r3,
           "rodada4": tabela_r4, "diagnostico_r5": diagnostico, "rodada5": tabela_r5, "rodada6_yamnet": tabela_r6, "rodada7_rawboost": tabela_r7, "rodada8_rawboost": tabela_r8, "condicoes_adversas": tabela_adversas, "microfone": tabela_microfone, "certeza": tabela_certeza,
           "intervencao_bordas": tabela_intervencao,
           "corte_tiro": tabela_corte, "teste": tabela_teste, "foco_seguranca": tabela_foco, "por_classe": por_classe}
if len(linhas_cv):
    tabelas.update({"validacao_cruzada": tabela_cv, "validacao_cruzada_resumo": resumo_cv})
for nome, tabela in tabelas.items():
    tabela.to_csv(os.path.join(PASTA_RESULTADOS, f"tab_{nome}.csv"))
arquivo_zip = shutil.make_archive("resultados", "zip", PASTA_RESULTADOS)
print("Gerado:", arquivo_zip, "com", len(os.listdir(PASTA_RESULTADOS)), "arquivos")
try:
    from google.colab import files
    files.download(arquivo_zip)
except ImportError:
    pass